<img src="https://raw.githubusercontent.com/microsoft/bioemu2/main/assets/emu.png" alt="BioEmu2 logo" width="90" align="right">

# BioEmu2 on Google Colab

**BioEmu2** (*biomolecular emulator*) is a foundation model from Microsoft Research AI for Science. It predicts folded structures, samples equilibrium ensembles, and propagates structures forward in time for proteins, nucleic acids, ligands and other chemical components.

This notebook installs BioEmu2 on a Colab GPU runtime and walks through the whole workflow:
**define a system → build an MSA → generate structures → view → analyse → simulate dynamics → download**.

[Source code](https://github.com/microsoft/bioemu2) · [Documentation](https://github.com/microsoft/bioemu2/tree/main/docs) · [Model card](https://github.com/microsoft/bioemu2/blob/main/MODEL_CARD.md) · [Checkpoints](https://huggingface.co/microsoft/bioemu2)

### Quick start

1. **Runtime → Change runtime type → select a GPU.** A free T4 works; L4, A100 or H100 runtimes are faster and fit larger systems.
2. **Runtime → Run all.** The defaults sample an equilibrium ensemble of *E. coli* adenylate kinase bound to the inhibitor Ap5A (PDB [1AKE](https://www.rcsb.org/structure/1AKE), closed state; the inhibitor is ligand `AP5` in the system table), then view, analyse and simulate it. The first run installs packages and downloads the model checkpoint, which takes several minutes.
3. **Use your own system:** change the form in a cell, then run that cell and the cells after it.

### Sections

| | Section | What it does |
|---|---|---|
| 1 | **Set up** | Check the GPU, install BioEmu2, choose storage, and start BioEmu2 |
| 2 | **Load the model** | Download the checkpoint and choose numerical precision |
| 3 | **Define the system** | From a PDB ID, sequences, an uploaded structure, a YAML file or an example; optional contacts |
| 4 | **MSA** | ColabFold search, your own A3M files, or single-sequence mode |
| 5 | **Generate structures** | Structure prediction or equilibrium sampling, with optional steering |
| 6 | **View** | Interactive 3D viewers |
| 7 | **Analyse** | RMSD, RMSF, radius of gyration, pairwise RMSD, PCA, secondary structure, physicality |
| 8 | **Simulate dynamics** | Time-ordered trajectories by propagating structures forward in time |
| 9 | **Save and download** | PDB export, zip archive, Google Drive |
| 10 | **Command line** | Run any `bioemu2` CLI command |
| 11 | **Troubleshooting** | Common problems and fixes |

**Tips**
- Cells with a form hide their code. Click **Show code** under a form to read or edit it.
- Each cell checks that the steps it depends on have run. If one has not, the cell tells you which section to run.
- Results are written to `/content/bioemu2/outputs/<job name>/`, or to Google Drive if you turn that on in cell 1.3.

> BioEmu2 is a research model. Generated structures are computational hypotheses, not experimental measurements; see the [model card](https://github.com/microsoft/bioemu2/blob/main/MODEL_CARD.md) for intended uses and limitations. This notebook uses the public [ColabFold](https://github.com/sokrypton/ColabFold) MMseqs2 server for MSAs: please run searches one at a time and cite ColabFold (Mirdita *et al.*, *Nature Methods* 2022) if you use them.

# 1. Set up the runtime

In [ ]:
#@title 1.1 Check the GPU runtime
#@markdown Confirms that the runtime has a supported NVIDIA GPU and Python version, and reports what is installed. No settings.
import os
import platform
import re
import shutil
import subprocess
import sys


def _nvidia_smi(*args):
    """Run nvidia-smi and return its stdout, or None when unavailable."""
    if shutil.which("nvidia-smi") is None:
        return None
    result = subprocess.run(["nvidia-smi", *args], capture_output=True, text=True)
    return result.stdout if result.returncode == 0 else None


def _preinstalled_torch():
    """Return (version, CUDA version) of the installed PyTorch without importing it here."""
    probe = (
        "import json, torch; "
        "print(json.dumps([torch.__version__, torch.version.cuda]))"
    )
    result = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if result.returncode != 0:
        return None, None
    import json

    version, cuda = json.loads(result.stdout.strip().splitlines()[-1])
    return version, cuda


gpu_csv = _nvidia_smi(
    "--query-gpu=name,memory.total,compute_cap,driver_version", "--format=csv,noheader,nounits"
)
if not gpu_csv:
    raise RuntimeError(
        "No NVIDIA GPU was found. BioEmu2 needs a GPU runtime: choose "
        "Runtime > Change runtime type > T4 GPU (or another GPU), then run this cell again."
    )
name, memory_mib, compute_cap, driver = [s.strip() for s in gpu_csv.splitlines()[0].split(",")]
header = _nvidia_smi() or ""
match = re.search(r"CUDA (?:UMD )?Version:\s*([0-9.]+)", header)
driver_cuda = float(match.group(1)) if match else 0.0
major, minor = (int(part) for part in compute_cap.split("."))
torch_version, torch_cuda = _preinstalled_torch()

GPU_INFO = {
    "name": name,
    "memory_gb": int(memory_mib) / 1024,
    "compute_capability": (major, minor),
    "driver": driver,
    "driver_cuda": driver_cuda,
    "gpu_count": len(gpu_csv.strip().splitlines()),
    "preinstalled_torch": torch_version,
    "preinstalled_torch_cuda": torch_cuda,
}

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk = shutil.disk_usage("/content" if os.path.isdir("/content") else "/")
print(f"GPU            {name}, {GPU_INFO['memory_gb']:.0f} GB, compute capability {compute_cap}")
print(f"NVIDIA driver  {driver} (supports CUDA up to {driver_cuda:g})")
print(f"Python         {platform.python_version()}")
print(f"PyTorch        {torch_version or 'not installed'}"
      + (f" (CUDA {torch_cuda})" if torch_cuda else ""))
print(f"Memory         {ram_gb:.0f} GB RAM, {disk.free / 1024**3:.0f} GB free disk")

problems = []
if sys.version_info < (3, 12):
    problems.append(
        f"BioEmu2 needs Python 3.12 or newer, but this runtime has {platform.python_version()}. "
        "Choose a newer runtime version under Runtime > Change runtime type."
    )
if (major, minor) < (7, 0):
    problems.append(f"{name} has compute capability {compute_cap}; BioEmu2 needs 7.0 or newer.")
if driver_cuda and driver_cuda < 12.0:
    problems.append(f"The NVIDIA driver supports CUDA {driver_cuda:g}; BioEmu2 needs CUDA 12 or newer.")
if problems:
    raise RuntimeError("\n".join(problems))
if sys.version_info >= (3, 13):
    print(f"\nNote: BioEmu2 passes its tests on Python {sys.version_info.major}.{sys.version_info.minor}; "
          "its benchmark metrics were evaluated on Python 3.12.")
if GPU_INFO["memory_gb"] < 20:
    print(f"\nNote: {GPU_INFO['memory_gb']:.0f} GB of GPU memory suits systems up to a few hundred "
          "residues. Use an L4, A100 or H100 runtime for large complexes.")
print("\nRuntime OK. Continue with cell 1.2.")

In [ ]:
#@title 1.2 Install BioEmu2
#@markdown Installs BioEmu2 with its GPU kernels (cuEquivariance) and the py3Dmol viewer. Installation takes a few minutes and is skipped on later runs in the same session.
source = "PyPI release"  #@param ["PyPI release", "GitHub (microsoft/bioemu2, main branch)", "Custom (enter below)"]
#@markdown For **Custom**, enter a pip requirement, git URL or local path, e.g. `bioemu2==0.1.0rc0`, `git+https://github.com/<user>/bioemu2@<branch>` or `/content/bioemu2`. The GPU extra is added automatically.
custom_source = ""  #@param {type:"string"}
#@markdown **GPU build.** *auto* keeps Colab's PyTorch when it is compatible (fastest) and otherwise installs a matching build. *CUDA 12.6* is the configuration the BioEmu2 authors tested on T4 GPUs and the only option for V100; it reinstalls PyTorch and does not support Blackwell GPUs (G4).
gpu_build = "auto"  #@param ["auto", "CUDA 13 (bioemu2[cuda])", "CUDA 12.6 (bioemu2[cuda12])"]
force_reinstall = False  #@param {type:"boolean"}

import importlib.metadata as metadata
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import sysconfig
import tempfile
import time
from pathlib import Path

if "GPU_INFO" not in globals():
    raise RuntimeError("Run cell 1.1 first.")

CUEQ_VERSION = "0.11.1"
CU126_INDEX = "https://download.pytorch.org/whl/cu126"


def _version_tuple(version):
    return tuple(int(part) for part in version.split("+")[0].split(".")[:2])


def _cuda_remedy(cc):
    """What to try when PyTorch or the GPU kernels do not work on this GPU."""
    if cc < (7, 5):
        return "Re-run cell 1.2 with gpu_build = 'CUDA 12.6' and force_reinstall ticked."
    advice = "Re-run cell 1.2 with gpu_build = 'CUDA 13' and force_reinstall ticked"
    if cc < (10, 0):
        advice += ", or with 'CUDA 12.6' (the configuration tested on T4 GPUs)"
    return advice + ". Runtime > Disconnect and delete runtime restores Colab's own PyTorch."


CUDA_REMEDY = _cuda_remedy(GPU_INFO["compute_capability"])


def _plan_install():
    """Choose extras and extra pip arguments for this GPU, driver and preinstalled PyTorch."""
    cc = GPU_INFO["compute_capability"]
    driver_cuda = GPU_INFO["driver_cuda"]
    torch_version = GPU_INFO["preinstalled_torch"]
    torch_cuda = GPU_INFO["preinstalled_torch_cuda"]
    cuda13_ok = cc >= (7, 5) and driver_cuda >= 13.0
    # PyTorch's CUDA 12.6 wheels stop at compute capability 9.0 (no Blackwell kernels).
    cuda126_ok = cc < (10, 0)
    cuda12 = {"extras": ["cuda12"], "packages": [], "args": ["--extra-index-url", CU126_INDEX],
              "reinstall_torch": False, "summary": "PyTorch 2.13 (CUDA 12.6) + cuEquivariance cu12"}
    if gpu_build.startswith("CUDA 13"):
        if not cuda13_ok:
            raise RuntimeError(
                f"CUDA 13 needs compute capability >= 7.5 and a driver supporting CUDA 13; "
                f"this runtime has {cc[0]}.{cc[1]} and CUDA {driver_cuda:g}. Choose another build."
            )
        return {"extras": ["cuda"], "packages": ["torch==2.13.0"], "args": [],
                "reinstall_torch": torch_cuda is not None and not torch_cuda.startswith("13"),
                "summary": "PyTorch 2.13 (CUDA 13) + cuEquivariance cu13"}
    if gpu_build.startswith("CUDA 12"):
        if not cuda126_ok:
            raise RuntimeError(f"PyTorch CUDA 12.6 builds have no kernels for compute capability "
                               f"{cc[0]}.{cc[1]} (Blackwell). Use gpu_build = 'auto' or 'CUDA 13'.")
        return cuda12
    # auto: keep a compatible preinstalled PyTorch and match the cuEquivariance CUDA flavour to it.
    if torch_version and torch_cuda and (2, 11) <= _version_tuple(torch_version) < (2, 14):
        cuda_major = int(torch_cuda.split(".")[0])
        if cuda_major == 13 and cuda13_ok:
            return {"extras": ["cuda"], "packages": [], "args": [], "reinstall_torch": False,
                    "summary": f"Colab's PyTorch {torch_version} + cuEquivariance cu13"}
        if cuda_major == 12:
            cueq = [f"cuequivariance=={CUEQ_VERSION}", f"cuequivariance-torch=={CUEQ_VERSION}",
                    f"cuequivariance-ops-torch-cu12=={CUEQ_VERSION}", f"cuequivariance-ops-cu12=={CUEQ_VERSION}"]
            return {"extras": [], "packages": cueq, "args": [], "reinstall_torch": False,
                    "summary": f"Colab's PyTorch {torch_version} + cuEquivariance cu12"}
    if cuda13_ok:
        return {"extras": ["cuda"], "packages": ["torch>=2.13,<2.14"], "args": [],
                "reinstall_torch": False, "summary": "PyTorch 2.13 (CUDA 13) + cuEquivariance cu13"}
    if not cuda126_ok:
        raise RuntimeError("This GPU needs a driver that supports CUDA 13. Choose a newer runtime "
                           "version under Runtime > Change runtime type.")
    return cuda12


def _normalize(name):
    return re.sub(r"[-_.]+", "-", name or "").lower()


def _requirement(extras):
    # The viz extra is left out on purpose: it upgrades ipywidgets to 8, which Colab does not support.
    extra = f"[{','.join(extras)}]" if extras else ""
    if source.startswith("PyPI"):
        return f"bioemu2{extra}>=0.1.0rc0"
    if source.startswith("GitHub"):
        return f"bioemu2{extra} @ git+https://github.com/microsoft/bioemu2.git"
    spec = custom_source.strip()
    if not spec:
        raise ValueError("Enter a pip requirement, git URL or path in custom_source.")
    if spec.startswith(("git+", "http://", "https://", "file:")):
        return f"bioemu2{extra} @ {spec}"
    if Path(spec).expanduser().exists():
        return f"bioemu2{extra} @ {Path(spec).expanduser().resolve().as_uri()}"
    from packaging.requirements import InvalidRequirement, Requirement

    try:
        parsed = Requirement(spec)
    except InvalidRequirement as error:
        raise ValueError(f"custom_source is not a pip requirement, URL or existing path: {spec!r}") from error
    if _normalize(parsed.name) != "bioemu2":
        raise ValueError("custom_source must install the bioemu2 package.")
    # Use this plan's CUDA flavour so cuda and cuda12 are never mixed; keep any other extras.
    parsed.extras = (set(parsed.extras) - {"cuda", "cuda12"}) | set(extras)
    return str(parsed)


def _run(command, env=None):
    """Run a command, echoing a condensed log; return (exit code, full output)."""
    print("$", " ".join(command))
    env = {**(env or os.environ), "NO_COLOR": "1"}
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, env=env)
    lines = []
    for raw in process.stdout:
        line = re.sub(r"\x1b\[[0-9;]*m", "", raw).rstrip()
        lines.append(line)
        if line.strip() and not line.lstrip().startswith(("+ ", "- ", "~ ", "Downloading", "Downloaded",
                                                          "Building", "Built", "warning: The `UV_")):
            print(line[:300])
    return process.wait(), lines


def _installed_versions():
    return {_normalize(dist.metadata["Name"]): dist.version for dist in metadata.distributions()}


VERIFY = r"""
import json, sys, torch
import bioemu2
from bioemu2.biomoa.ops import triangle_multiplication
from importlib.metadata import version
info = {"bioemu2": version("bioemu2"), "torch": torch.__version__, "torch_cuda": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(), "cuequivariance": triangle_multiplication.HAS_CUEQ}
if torch.cuda.is_available():
    info["matmul"] = bool(torch.isfinite(torch.rand(256, 256, device="cuda") @ torch.rand(256, 256, device="cuda")).all())
    try:
        import triton, triton.language as tl

        @triton.jit
        def _add(x, y, out, n, BLOCK: tl.constexpr):
            offsets = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
            mask = offsets < n
            tl.store(out + offsets, tl.load(x + offsets, mask=mask) + tl.load(y + offsets, mask=mask), mask=mask)

        a = torch.rand(1000, device="cuda"); b = torch.rand(1000, device="cuda"); c = torch.empty_like(a)
        _add[(4,)](a, b, c, 1000, BLOCK=256)
        info["triton"] = bool(torch.allclose(c, a + b))
    except Exception as error:
        info["triton"] = False
        info["triton_error"] = f"{type(error).__name__}: {error}"[-500:]
print(json.dumps(info))
"""


def _verify():
    # Triton reads kernel sources from disk, so the check runs from a file rather than `python -c`.
    script = Path(tempfile.gettempdir()) / "bioemu2_colab_verify.py"
    script.write_text(VERIFY)
    result = subprocess.run([sys.executable, str(script)], capture_output=True, text=True)
    if result.returncode != 0:
        return None, result.stderr[-3000:]
    return json.loads(result.stdout.strip().splitlines()[-1]), ""


# Triton locates libcuda through the linker cache; register Colab's driver directory if needed.
if Path("/usr/lib64-nvidia").is_dir() and shutil.which("ldconfig"):
    cache = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
    if "libcuda.so" not in cache:
        subprocess.run(["ldconfig", "/usr/lib64-nvidia"], check=False)

plan = _plan_install()
requirement = _requirement(plan["extras"])
marker = Path.home() / ".bioemu2_colab_install.json"
signature = {"requirement": requirement, "plan": plan, "python": sys.version}
already = (marker.exists() and json.loads(marker.read_text()) == signature
           and importlib.util.find_spec("bioemu2") is not None)

if already and not force_reinstall:
    print(f"BioEmu2 is already installed in this session ({plan['summary']}).")
else:
    print(f"Plan: {requirement}\n      {plan['summary']}\n")
    start = time.time()
    # Triton compiles small C helpers at run time and needs the Python headers.
    include_dir = Path(sysconfig.get_paths()["include"])
    if not (include_dir / "Python.h").exists() and shutil.which("apt-get"):
        version = f"{sys.version_info.major}.{sys.version_info.minor}"
        print(f"Installing Python {version} headers for Triton...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(["apt-get", "-qq", "install", "-y", f"python{version}-dev"], check=False)

    before = _installed_versions()
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=False)
    targets = [requirement, *plan["packages"], "py3Dmol"]
    # Colab may point uv's constraint variables at files that do not exist; they would make uv fail.
    uv_env = {k: v for k, v in os.environ.items()
              if k not in ("UV_CONSTRAINT", "UV_BUILD_CONSTRAINT") and not (k == "UV_PRERELEASE" and not v)}
    log = []
    if shutil.which("uv"):
        command = ["uv", "pip", "install", "--python", sys.executable, *targets, *plan["args"]]
        if plan["args"]:
            command += ["--index-strategy", "unsafe-best-match"]
        if plan["reinstall_torch"]:
            command += ["--reinstall-package", "torch"]
        if force_reinstall:
            command += ["--reinstall-package", "bioemu2"]
        code, log = _run(command, env=uv_env)
    else:
        code = 1
    if code != 0:
        print("\nuv did not succeed; retrying with pip...")
        pip = [sys.executable, "-m", "pip", "install"]
        if any("externally managed" in line.lower() for line in log):
            pip.append("--break-system-packages")
        env = {k: v for k, v in os.environ.items() if k not in ("PIP_CONSTRAINT", "UV_CONSTRAINT")}
        # Never force-reinstall the whole dependency tree: that would replace Colab's torch and numpy.
        steps = []
        if plan["reinstall_torch"]:
            steps.append([*pip, "--force-reinstall", *[p for p in plan["packages"] if p.startswith("torch")]])
        steps.append([*pip, *targets, *plan["args"]])
        if force_reinstall:
            steps.append([*pip, "--force-reinstall", "--no-deps", requirement, *plan["args"]])
        for command in steps:
            code, log = _run(command, env=env)
            if code != 0:
                break
    if code != 0:
        print("\n".join(log[-40:]))
        raise RuntimeError("Installation failed; see the log above and section 11 (Troubleshooting).")
    importlib.invalidate_caches()

    info, error = _verify()
    if info is None:
        print(error)
        raise RuntimeError(f"BioEmu2 installed but failed to import. {CUDA_REMEDY} "
                           "See also section 11 (Troubleshooting).")
    print(f"\nInstalled in {time.time() - start:.0f} s: " + json.dumps(info))
    if not info.get("cuda_available"):
        raise RuntimeError(f"PyTorch cannot see the GPU. {CUDA_REMEDY}")
    if not info.get("cuequivariance"):
        print("Warning: the cuEquivariance kernels failed to import. The model then works only with "
              "kernels = 'pure PyTorch' in section 2. " + CUDA_REMEDY)
    if info.get("triton") is False:
        print(f"Warning: the Triton self-test failed ({info.get('triton_error', 'unknown error')}); "
              "see section 11 (Troubleshooting).")
    if info.get("cuequivariance"):
        marker.write_text(json.dumps(signature))  # Left unwritten otherwise, so the next run retries.

    # Packages that this kernel already imported must be reloaded from a fresh process. bioemu2
    # itself counts as changed when it was imported, because a reinstall from GitHub or a local
    # path can change the code without changing the version number.
    changed = {name for name, version in _installed_versions().items() if before.get(name) != version}
    if any(module == "bioemu2" or module.startswith("bioemu2.") for module in sys.modules):
        changed.add("bioemu2")
    loaded = {
        module
        for module, dists in metadata.packages_distributions().items()
        if module in sys.modules and any(_normalize(dist) in changed for dist in dists)
    }
    if loaded:
        print(f"\nUpgraded packages that were already loaded: {', '.join(sorted(loaded))}.")
        print("Restarting the runtime to load them. When it reconnects, continue with cell 1.3 "
              "(or use Runtime > Run all again; the installation will be skipped).")
        time.sleep(1)
        os.kill(os.getpid(), 9)
print("Continue with cell 1.3.")

In [ ]:
#@title 1.3 Storage, caches and access tokens
#@markdown **Google Drive.** Store outputs and caches (model checkpoint, MSAs, ligand definitions) on Google Drive so they persist between sessions and later sessions skip the downloads. Turn this on before defining the system in 3.1; if a job already exists, its new runs go to Drive from then on.
save_to_google_drive = False  #@param {type:"boolean"}
drive_folder = "bioemu2"  #@param {type:"string"}
#@markdown **Trunk-embedding cache.** Re-use trunk embeddings on disk across sessions for identical inputs. Within a session the notebook re-uses them in memory anyway. Most useful with Google Drive. Takes effect when the model is (re)loaded in section 2.
cache_trunk_embeddings = False  #@param {type:"boolean"}
#@markdown **Hugging Face token.** Only needed if the checkpoint download requires authentication: add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) and grant this notebook access.
use_hf_token_secret = True  #@param {type:"boolean"}
log_level = "WARNING"  #@param ["WARNING", "INFO", "DEBUG"]

import logging
import os
import shutil
import sys
from pathlib import Path

if save_to_google_drive:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive") / drive_folder
else:
    BASE_DIR = Path("/content/bioemu2") if Path("/content").is_dir() else Path.home() / "bioemu2"

OUTPUT_ROOT = BASE_DIR / "outputs"
CACHE_ROOT = BASE_DIR / "cache"
UPLOAD_DIR = Path("/content/bioemu2/uploads") if Path("/content").is_dir() else Path.home() / "bioemu2/uploads"
for directory in (OUTPUT_ROOT, CACHE_ROOT, UPLOAD_DIR):
    directory.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = CACHE_ROOT / "huggingface"
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR)
if "huggingface_hub" in sys.modules:  # It reads HF_HUB_CACHE once, when 1.4 imports it.
    sys.modules["huggingface_hub"].constants.HF_HUB_CACHE = str(HF_CACHE_DIR)
os.environ["BIOEMU2_MSA_CACHE"] = str(CACHE_ROOT / "colabfold_msa")
os.environ["BIOEMU2_CCD_CACHE"] = str(CACHE_ROOT / "ccd")
os.environ["BIOEMU2_TEMPLATE_CACHE"] = str(CACHE_ROOT / "templates")
if cache_trunk_embeddings:
    os.environ["BIOEMU2_EMBEDS_CACHE"] = str(CACHE_ROOT / "embeddings")
else:
    os.environ.pop("BIOEMU2_EMBEDS_CACHE", None)
LOG_LEVEL = log_level
logging.getLogger().setLevel(getattr(logging, log_level))

token_status = "not used"
if use_hf_token_secret:
    try:
        from google.colab import userdata

        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        token_status = "loaded from the HF_TOKEN secret"
    except Exception as error:  # Missing secret, access not granted, or not running on Colab.
        token_status = f"not found ({type(error).__name__}); downloads will be anonymous"

print(f"Outputs        {OUTPUT_ROOT}")
print(f"Caches         {CACHE_ROOT}")
print(f"Embedding cache {'on' if cache_trunk_embeddings else 'off'}")
print(f"HF token       {token_status}")

# On a re-run, send the current job's new runs to the new output folder.
if globals().get("JOB_NAME") and globals().get("JOB_DIR") is not None and JOB_DIR.parent != OUTPUT_ROOT:
    old_job_dir, JOB_DIR = JOB_DIR, OUTPUT_ROOT / JOB_NAME
    for sub in ("input", "msa"):
        if (old_job_dir / sub).is_dir():
            shutil.copytree(old_job_dir / sub, JOB_DIR / sub, dirs_exist_ok=True)
    print(f"\nNew runs of job '{JOB_NAME}' will be saved in {JOB_DIR}; earlier runs stay in {old_job_dir}.")
if globals().get("MODEL") is not None and \
        (getattr(MODEL, "embedding_cache_dir", None) is not None) != cache_trunk_embeddings:
    print("Re-run section 2 for the trunk-embedding cache setting to take effect.")

In [ ]:
#@title 1.4 Start BioEmu2
#@markdown Imports BioEmu2, checks the GPU stack and defines the helper functions used by the rest of the notebook. No settings.
import gc
import hashlib
import json
import logging
import math
import re
import shutil
import sys
import tempfile
import time
import traceback
import warnings
import zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path

if "OUTPUT_ROOT" not in globals():
    raise RuntimeError("Run cell 1.3 first.")

import matplotlib.pyplot as plt
import mdtraj
import numpy as np
import py3Dmol
import torch
from IPython.display import HTML, display

import bioemu2
from bioemu2 import (
    MSA,
    Atom,
    Bond,
    Contact,
    EmuSession,
    System,
    check_physicality,
    get_msa,
    load_model,
    load_system_and_contacts,
    map_cif_atom_order,
    save_frames,
    simulate_trajectory_with_embeddings,
)
from bioemu2.data_interface import ContactToken, Protein
from bioemu2.paths import EXAMPLES_DIR
from bioemu2.sampling import sampler_from_preset

logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(levelname)s %(name)s: %(message)s",
                    force=True)
for noisy in ("httpx", "urllib3", "filelock", "numba", "matplotlib"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message=".*Unlikely unit cell vectors.*")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------------------------- notebook state
# Each section stores its result in one of these globals. Later sections check that the
# results they need exist and were computed from the current inputs.
MODEL = None
MODEL_INFO = {}
SYSTEM = None
SYSTEM_TOKENS = None
CONTACTS = None
MSA_OBJ = None
MSA_FOR = None            # system_key() of the system the MSA was built for
REFERENCE = None          # {(chain, residue number): CA/C4' coordinates in nm} from the input structure
REFERENCE_AUTHOR = None   # {(chain, residue number): author chain and residue number} from the input file
REFERENCE_FILE = None     # Path of the input structure, when there is one
JOB_NAME = None
JOB_DIR = None
EMBEDDINGS = {}           # mode -> (input key, TrunkEmbeddings); at most one mode at a time
RUNS = {}                 # run name -> run record (see record_run); the newest run is last
LAST_RUN = None
if "CUDA_REMEDY" not in globals():
    CUDA_REMEDY = "Re-run cell 1.2 with another gpu_build and force_reinstall ticked."

_SECTION_OF = {"MODEL": "2", "SYSTEM": "3.1", "MSA_OBJ": "4"}


def system_key(system):
    """A content key for a system: its YAML form. Runs and MSAs are tied to it."""
    return None if system is None else str(system)


def require(*names):
    """Raise a helpful error if a prerequisite section has not been run."""
    missing = [name for name in names if globals().get(name) is None]
    if missing:
        sections = ", ".join(sorted({_SECTION_OF.get(name, "?") for name in missing}))
        raise RuntimeError(f"Run section {sections} first (missing: {', '.join(missing)}).")
    if "MSA_OBJ" in names and MSA_FOR != system_key(SYSTEM):
        raise RuntimeError("The MSA belongs to a previous system. Run section 4 again.")


def drop_sessions():
    """Forget the sampling session and embeddings left by the last sampling cell."""
    for name in ("session", "embeddings"):
        globals().pop(name, None)


def reset_downstream(*, msa=True):
    """Forget results that depend on the system, contacts or MSA."""
    global MSA_OBJ, MSA_FOR
    drop_sessions()
    EMBEDDINGS.clear()
    if msa:
        MSA_OBJ, MSA_FOR = None, None
    free_gpu_memory()


def free_gpu_memory():
    """Release cached GPU memory, including tensors held by the traceback of a failed cell."""
    exception, seen = getattr(sys, "last_value", None), set()
    while exception is not None and id(exception) not in seen:
        seen.add(id(exception))
        traceback.clear_frames(exception.__traceback__)
        exception = exception.__cause__ or exception.__context__
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def gpu_memory_report():
    if not torch.cuda.is_available():
        return "CPU"
    used = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    return f"{used:.1f} GB in use, peak {peak:.1f} GB of {total:.0f} GB"


def is_out_of_memory(error):
    return isinstance(error, torch.cuda.OutOfMemoryError) or "out of memory" in str(error).lower()


def out_of_memory_advice(stage):
    tips = []
    if MODEL_INFO.get("precision") != "bf16-mixed":
        tips.append("precision = 'bf16-mixed' (section 2)")
    if str(MODEL_INFO.get("kernels", "")).startswith("memory"):
        tips.append("a smaller tile_size (128 or 64) for the memory-saving kernels (section 2)")
    else:
        tips.append("kernels = 'memory-saving (large systems)' (section 2)")
    tips += ["a smaller max_msa_rows (section 4)", "fewer chains or ligands (section 3)",
             "a runtime with more GPU memory"]
    return f"Out of GPU memory while {stage}. Try: " + "; ".join(tips) + "."


# ----------------------------------------------------------------------------- small utilities
def split_list(text):
    """Split a form field of names or numbers on commas, semicolons, colons, slashes or whitespace."""
    return [item for item in re.split(r"[\s,;:/]+", text.strip()) if item]


def slug(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", text.strip()).strip("_") or "job"


def unique_dir(path):
    """Return ``path`` or the first free ``path_2``, ``path_3``, ..."""
    path = Path(path)
    if not path.exists():
        return path
    for index in range(2, 100_000):
        candidate = path.with_name(f"{path.name}_{index}")
        if not candidate.exists():
            return candidate
    raise RuntimeError(f"No free directory name next to {path}.")


def show_table(rows, headers):
    """Display a list of rows as an HTML table."""
    head = "".join(f"<th style='text-align:left;padding:2px 10px'>{h}</th>" for h in headers)
    body = "".join(
        "<tr>" + "".join(f"<td style='padding:2px 10px'>{cell}</td>" for cell in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))


def clean_upload_name(name):
    """Undo the ' (1)' that browsers and Colab add to repeated uploads: 'x (1).cif' -> 'x.cif'."""
    return re.sub(r" \(\d+\)", "", Path(name).name)


def get_files(path_field, prompt, *, multiple=False):
    """Return paths from a comma-separated form field, or open Colab's upload dialog.

    Each upload goes into its own new folder, so files with the same name never overwrite each other.
    """
    if path_field.strip():
        paths = [Path(p.strip()).expanduser() for p in path_field.split(",") if p.strip()]
        for path in paths:
            if not path.is_file():
                raise FileNotFoundError(f"File not found: {path}")
        return paths
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError("File upload needs Google Colab; enter a file path instead.") from error
    print(prompt)
    incoming = Path(tempfile.mkdtemp(dir=UPLOAD_DIR, prefix="incoming_"))
    try:
        uploaded = files.upload(target_dir=str(incoming))
    except TypeError:  # Older Colab versions have no target_dir.
        uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded.")
    folder = Path(tempfile.mkdtemp(dir=UPLOAD_DIR, prefix=datetime.now().strftime("%Y%m%d_%H%M%S_")))
    paths = []
    for name, data in uploaded.items():
        path = folder / clean_upload_name(name)
        path.write_bytes(data)
        paths.append(path)
    shutil.rmtree(incoming, ignore_errors=True)
    if not multiple and len(paths) > 1:
        print(f"Using {paths[0].name}; ignoring the other uploaded files.")
        paths = paths[:1]
    return paths


# ----------------------------------------------------------------------------- runs
def record_run(name, *, kind, mode, frames, directory, report=None, settings=None, trajectories=None):
    """Store a run so that later sections can view, analyse, propagate and export it.

    Runs are named after their folder, or 'job/folder' if another job already has a run of
    that name. The newest run is always last. Each run keeps the system and input structure
    it was generated from.
    """
    global LAST_RUN
    directory = Path(directory)
    key = name
    if key in RUNS and RUNS[key]["dir"] != directory:
        key = f"{directory.parent.name}/{name}"
    RUNS.pop(key, None)
    RUNS[key] = {"name": key, "kind": kind, "mode": mode, "frames": frames, "dir": directory,
                 "report": report, "settings": settings or {}, "trajectories": trajectories,
                 "job": JOB_NAME, "system_key": system_key(SYSTEM), "reference": REFERENCE,
                 "reference_author": REFERENCE_AUTHOR, "reference_file": REFERENCE_FILE}
    LAST_RUN = key
    return RUNS[key]


def resolve_run(name, kinds=None):
    """Return the run record for a form value: a run name, or 'latest' (of the given kinds).

    'latest' means the newest run of the current system; runs of earlier systems are used
    only if there is none, with a note.
    """
    if not RUNS:
        raise RuntimeError("No results yet. Run section 5 first.")
    current = system_key(SYSTEM)
    wanted = [key for key, run in RUNS.items() if kinds is None or run["kind"] in kinds]
    kind_text = " or ".join(kinds) if kinds else "any"
    if name.strip().lower() in ("", "latest"):
        mine = [key for key in wanted if RUNS[key]["system_key"] == current]
        if mine:
            return RUNS[mine[-1]]
        if not wanted:
            raise RuntimeError(f"No {kind_text} run yet.")
        print(f"Note: no {kind_text} run of the current system yet; using '{wanted[-1]}' "
              "from an earlier system.")
        return RUNS[wanted[-1]]
    key = name.strip()
    if key not in RUNS:
        matches = [k for k in RUNS if k.split("/")[-1] == key]
        if len(matches) > 1:
            matches = [k for k in matches if RUNS[k]["job"] == JOB_NAME] or matches
        if len(matches) != 1:
            raise KeyError(f"No unique run named {key!r}. Available runs: {', '.join(RUNS)}")
        key = matches[0]
    if kinds is not None and RUNS[key]["kind"] not in kinds:
        raise ValueError(f"Run {key!r} is a {RUNS[key]['kind']} run; choose a {kind_text} run.")
    return RUNS[key]


def list_runs():
    rows = [[name, run["kind"], run["mode"], run["frames"].n_frames, run["job"],
             "" if run["system_key"] == system_key(SYSTEM) else "earlier system"]
            for name, run in RUNS.items()]
    show_table(rows, ["run", "kind", "mode", "frames", "job", ""])


def select_frames(run, which):
    """Return (trajectory, run frame numbers) for 'all frames' or 'physically passing frames'."""
    trajectory = run["frames"]
    report = run.get("report")
    if which.startswith("physically") and report is not None and report.checked:
        keep = np.flatnonzero(report.passed)
        if len(keep) == 0:
            print("No frame passed the physicality checks; using all frames.")
            return trajectory, np.arange(trajectory.n_frames)
        return trajectory[keep], keep
    return trajectory, np.arange(trajectory.n_frames)


def physicality_summary(report, n_frames=None):
    """One line on a physicality report, saying when nothing (or only protein) was checked."""
    if report is None or not report.checked or report.mode == "no_filter":
        return "not checked" + ("" if report is None or report.mode == "no_filter"
                                else " (only protein chains are checked)")
    passed = int(report.passed.sum())
    text = f"{passed}/{len(report.passed) if n_frames is None else n_frames} pass '{report.mode}'"
    if any(not isinstance(e, Protein) for e in SYSTEM.entities):
        text += " (protein chains only)"
    reasons = Counter(r for r in report.reasons if r)
    if reasons:
        text += "; failures: " + ", ".join(f"{k} ({v})" for k, v in reasons.most_common(5))
    return text


def write_system_yaml(path, system, contacts):
    """Write a system YAML that bioemu2 (CLI or load_system_and_contacts) can read back."""
    import yaml

    data = system.model_dump(exclude_none=True)
    if contacts:
        data["contacts"] = [contact.model_dump(exclude_none=True) for contact in contacts]
    path.write_text(yaml.safe_dump(data, sort_keys=False))


def embeddings_for(mode):
    """Trunk embeddings for the current model, system, MSA and contacts.

    Only one set is kept on the GPU: switching between structure and equilibrium mode
    recomputes them.
    """
    require("MODEL", "SYSTEM", "MSA_OBJ")
    key = (id(MODEL), system_key(SYSTEM), id(MSA_OBJ), repr(CONTACTS))
    cached = EMBEDDINGS.get(mode)
    if cached is not None and cached[0] == key:
        return cached[1]
    del cached
    drop_sessions()
    EMBEDDINGS.clear()
    free_gpu_memory()
    print(f"Computing {mode} trunk embeddings ({SYSTEM_TOKENS} tokens)...", end=" ", flush=True)
    start = time.time()
    out_of_memory = False
    try:
        embeddings = MODEL.embed(SYSTEM, msa=MSA_OBJ, mode=mode, contacts=CONTACTS)
    except Exception as error:
        if not is_out_of_memory(error):
            raise
        out_of_memory = True  # Leaving the except block releases the failed call's tensors.
    if out_of_memory:
        free_gpu_memory()
        raise RuntimeError(out_of_memory_advice("computing trunk embeddings"))
    print(f"done in {time.time() - start:.0f} s ({gpu_memory_report()}).")
    EMBEDDINGS[mode] = (key, embeddings)
    return embeddings


# ----------------------------------------------------------------------------- systems
def system_entities_table(system):
    """Rows describing each chain: canonical ID, author ID, type, size and sequence."""
    rows = []
    author = system.asym_id_to_author_id or {}
    for entity in system.entities:
        kind = type(entity).__name__
        if hasattr(entity, "sequence"):
            sequence = entity.sequence
            size = f"{len(sequence)} residues"
            detail = sequence if len(sequence) <= 60 else sequence[:57] + "..."
            mods = getattr(entity, "mods", None) or []
            if mods:
                detail += f" ({len(mods)} modified)"
        elif kind == "Ligand":
            size = "ligand"
            detail = entity.ccd_code or entity.smiles or entity.inchi or ""
        else:
            monomers = getattr(entity, "mon_seq", []) or []
            size = f"{len(monomers)} monomers"
            detail = " ".join(getattr(m, "ccd_code", "") or "" for m in monomers)
        for chain in entity.asym_ids:
            rows.append([chain, author.get(chain, "–"), kind, size, f"<code>{detail}</code>"])
    return rows


def count_tokens(system):
    """Number of model tokens: one per standard residue, one per atom of ligands and modified residues."""
    from bioemu2.structure.preparation import prepare_sampling_inputs

    try:
        _, batch = prepare_sampling_inputs(system, device=torch.device("cpu"))
        return int(batch.num_tokens.item())
    except Exception as error:  # Fall back to a residue count if preparation fails here.
        print(f"Could not count tokens exactly ({error}); using residue counts.")
        return sum(len(getattr(e, "sequence", "") or "x") * len(e.asym_ids) for e in system.entities)


def max_batch_estimate(num_tokens):
    """BioEmu2's own rule of thumb for the largest denoising batch that fits in GPU memory."""
    if not torch.cuda.is_available():
        return 8
    total = torch.cuda.get_device_properties(0).total_memory
    return int(0.8 * total / (0.3e6 * num_tokens) - num_tokens / 75)


def auto_batch_size(num_samples, sampler_name):
    """Pick a batch size that leaves memory headroom; steering needs gradients and more memory."""
    num_tokens = SYSTEM_TOKENS or count_tokens(SYSTEM)
    fraction = {"edm": 0.5, "edm_gd": 0.25, "edm_fk": 0.25, "edm_fkc": 0.125,
                "fk_resampling_only": 0.5, "fkc_smc": 0.4}.get(sampler_name, 0.25)
    return max(1, min(num_samples, 64, int(max_batch_estimate(num_tokens) * fraction)))


def _chain_name(position):
    """Chain ID that System.from_description gives the argument at ``position``: A..Z, AA, AB, ..."""
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    width, offset = 1, position
    while offset >= 26**width:
        offset -= 26**width
        width += 1
    name = ""
    for _ in range(width):
        offset, digit = divmod(offset, 26)
        name = letters[digit] + name
    return name


def _peptide_bonded(first, second, cutoff=2.0):
    c, n = first[first.atom_name == "C"], second[second.atom_name == "N"]
    return bool(len(c) and len(n) and np.linalg.norm(c.coord[0] - n.coord[0]) < cutoff)


def structure_file_to_system(path, *, include_ligands=True, exclude_ligands=()):
    """Build a System from the resolved residues of a PDB-format (or metadata-poor mmCIF) file.

    Returns the system, its reference {(chain, residue number): CA/C4' coordinates in nm}, the
    author numbering {(chain, residue number): file chain and residue}, and the cysteine SG atoms
    [(chain, residue number, coordinates in Å)]. Unresolved residues are absent from such files,
    so chain breaks are joined. Free amino acids (HETATM residues not peptide-bonded to the
    chain) become ligands.
    """
    import biotite.structure as struc
    from biotite.sequence import ProteinSequence
    from biotite.structure.io import load_structure

    atoms = load_structure(str(path), model=1)
    atoms = atoms[atoms.element != "H"]
    proteins, rnas, dnas, ligands = [], [], [], []
    file_chain, ligand_chains, rows, gaps = {}, [], [], []
    for chain_atoms in struc.chain_iter(atoms):
        chain_id = str(chain_atoms.chain_id[0])
        residues = list(struc.residue_iter(chain_atoms))
        amino = [bool(struc.filter_amino_acids(r).any()) for r in residues]
        nucleic = [bool(struc.filter_nucleotides(r).any()) for r in residues]
        polymer, extra_ligands = [], []
        for i, residue in enumerate(residues):
            if amino[i] and residue.hetero[0]:
                linked = (i > 0 and amino[i - 1] and _peptide_bonded(residues[i - 1], residue)) or (
                    i + 1 < len(residues) and amino[i + 1] and _peptide_bonded(residue, residues[i + 1]))
                (polymer if linked else extra_ligands).append(residue)
            elif amino[i] or (nucleic[i] and not residue.hetero[0]):
                polymer.append(residue)
            elif residue.hetero[0]:
                extra_ligands.append(residue)
        protein_part = [r for r in polymer if struc.filter_amino_acids(r).any()]
        nucleic_part = [r for r in polymer if struc.filter_nucleotides(r).any()]
        kind = None
        if protein_part and len(protein_part) >= len(nucleic_part):
            letters = []
            for residue in protein_part:
                try:
                    letters.append(ProteinSequence.convert_letter_3to1(residue.res_name[0]))
                except KeyError:
                    letters.append("X")
            kind, chain_residues = "protein", protein_part
            proteins.append("".join(letters))
        elif nucleic_part:
            names = [r.res_name[0].strip() for r in nucleic_part]
            is_dna = any(n.startswith("D") for n in names)
            sequence = "".join((n[-1] if n[-1] in "ACGTU" else "N") for n in names)
            kind, chain_residues = ("dna" if is_dna else "rna"), nucleic_part
            (dnas if is_dna else rnas).append(sequence.replace("U", "T") if is_dna else sequence.replace("T", "U"))
        if kind:
            index = {"protein": len(proteins), "rna": len(rnas), "dna": len(dnas)}[kind] - 1
            file_chain[(kind, index)] = chain_id
            numbers = [int(r.res_id[0]) for r in chain_residues]
            if any(b - a > 1 for a, b in zip(numbers, numbers[1:])):
                gaps.append(chain_id)
            for ordinal, residue in enumerate(chain_residues, start=1):
                rows.append((kind, index, ordinal, residue))
        if include_ligands:
            for residue in extra_ligands:
                code3 = residue.res_name[0].strip().upper()
                if code3 not in ("HOH", "WAT", "DOD") and code3 not in exclude_ligands:
                    ligands.append(code3)
                    ligand_chains.append(chain_id)
    if not (proteins or rnas or dnas or ligands):
        raise ValueError(f"No protein, nucleic-acid or ligand residues found in {path}.")
    system = System.from_description(proteins=proteins, rna=rnas, dna=dnas, ligand_ccd_codes=ligands)
    # from_description names chains A, B, ..., Z, AA, AB, ... in argument order (proteins, RNA,
    # DNA, ligands). Identical sequences are then merged into one entity, which reorders
    # system.chain_ids, so derive each chain's ID from its position in the arguments.
    order = [("protein", i) for i in range(len(proteins))] + [("rna", i) for i in range(len(rnas))] \
        + [("dna", i) for i in range(len(dnas))]
    chain_of = {key: _chain_name(position) for position, key in enumerate(order)}
    author = {chain_of[key]: file_chain[key] for key in order}
    author.update({_chain_name(len(order) + j): c for j, c in enumerate(ligand_chains)})
    system = System(system.entities, bonds=system.bonds, asym_id_to_author_id=author)
    reference, author_numbers, cysteines = {}, {}, []
    for kind, index, ordinal, residue in rows:
        chain = chain_of[(kind, index)]
        author_numbers[(chain, ordinal)] = f"{file_chain[(kind, index)]}:{residue.res_id[0]}{residue.ins_code[0]}"
        for atom_name in ("CA", "C4'"):
            hit = residue[residue.atom_name == atom_name]
            if len(hit):
                reference[(chain, ordinal)] = hit.coord[0] / 10.0
        sg = residue[(residue.atom_name == "SG") & (residue.res_name == "CYS")]
        if len(sg):
            cysteines.append((chain, ordinal, sg.coord[0]))
    if gaps:
        print(f"Note: file chain(s) {', '.join(gaps)} have missing residues; the gaps are joined in the "
              "sequence. Use an mmCIF file with full sequence records to keep them.")
    return system, reference, author_numbers, cysteines


def cysteines_from_cif(path, chain_ids):
    """Cysteine SG atoms [(label chain, label residue number, coordinates in Å)] of an mmCIF file."""
    import biotite.structure.io.pdbx as pdbx

    atoms = pdbx.get_structure(pdbx.CIFFile.read(str(path)), model=1, use_author_fields=False)
    sg = atoms[(atoms.res_name == "CYS") & (atoms.atom_name == "SG") & np.isin(atoms.chain_id, list(chain_ids))]
    return [(str(c), int(r), xyz) for c, r, xyz in zip(sg.chain_id, sg.res_id, sg.coord)]


def add_disulfides(system, cysteines, cutoff=2.5):
    """Declare S–S bonds between cysteine SG atoms closer than ``cutoff`` Å.

    Steering samplers keep declared disulfides at bond length; undeclared ones are pushed apart.
    """
    sequence_of = {chain: entity.sequence for entity in system.entities if isinstance(entity, Protein)
                   for chain in entity.asym_ids}
    modified = {(chain, mod.res_id) for entity in system.entities if isinstance(entity, Protein)
                for mod in (entity.mods or []) for chain in entity.asym_ids}
    usable = [(c, r, xyz) for c, r, xyz in cysteines if c in sequence_of and 0 < r <= len(sequence_of[c])
              and sequence_of[c][r - 1] == "C" and (c, r) not in modified]
    existing = {frozenset({(b.atom1.asym_id, b.atom1.res_id, b.atom1.name),
                           (b.atom2.asym_id, b.atom2.res_id, b.atom2.name)}) for b in system.bonds}
    bonds, labels = list(system.bonds), []
    for i in range(len(usable)):
        for j in range(i + 1, len(usable)):
            (c1, r1, x1), (c2, r2, x2) = usable[i], usable[j]
            pair = frozenset({(c1, r1, "SG"), (c2, r2, "SG")})
            if np.linalg.norm(np.asarray(x1) - np.asarray(x2)) < cutoff and pair not in existing:
                bonds.append(Bond(atom1=Atom(name="SG", res_name="CYS", res_id=r1, asym_id=c1),
                                  atom2=Atom(name="SG", res_name="CYS", res_id=r2, asym_id=c2)))
                existing.add(pair)
                labels.append(f"{c1}:C{r1}–{c2}:C{r2}")
    if not labels:
        return system, labels
    return System(system.entities, bonds=bonds, asym_id_to_author_id=system.asym_id_to_author_id), labels


# ----------------------------------------------------------------------------- reference structures
_REPRESENTATIVE = ("CA", "C4'")


def representative_atoms(topology):
    """Map (chain ID, residue number) to the index of each residue's CA or C4' atom."""
    keys = {}
    for atom in topology.atoms:
        residue = atom.residue
        if atom.name in _REPRESENTATIVE and (residue.is_protein or residue.is_nucleic):
            keys[(residue.chain.chain_id, residue.resSeq)] = atom.index
    return keys


def reference_from_cif(path):
    """CA/C4' coordinates (nm) and author numbering, keyed by (label_asym_id, label_seq_id)."""
    import biotite.structure.io.pdbx as pdbx

    cif = pdbx.CIFFile.read(str(path))
    label = pdbx.get_structure(cif, model=1, use_author_fields=False)
    author = pdbx.get_structure(cif, model=1, use_author_fields=True)
    mask = np.isin(label.atom_name, _REPRESENTATIVE) & (label.res_id > 0)
    coords = {(str(c), int(r)): xyz / 10.0
              for c, r, xyz in zip(label.chain_id[mask], label.res_id[mask], label.coord[mask])}
    numbers = {}
    if len(author) == len(label):
        numbers = {(str(c), int(r)): f"{ac}:{ar}{ai}" for c, r, ac, ar, ai in zip(
            label.chain_id[mask], label.res_id[mask], author.chain_id[mask], author.res_id[mask],
            author.ins_code[mask])}
    return coords, numbers


def representative_indices(trajectory):
    return np.array(sorted(representative_atoms(trajectory.topology).values()), dtype=int)


def matched_reference(trajectory, reference=None):
    """Sample atom indices and a one-frame reference trajectory for residues in both, or None."""
    reference = REFERENCE if reference is None else reference
    if not reference:
        return None
    keys = representative_atoms(trajectory.topology)
    common = [key for key in keys if key in reference]
    if len(common) < 3:
        return None
    indices = np.array([keys[key] for key in common])
    target = mdtraj.Trajectory(np.stack([reference[key] for key in common])[None],
                               trajectory.topology.subset(indices))
    return indices, target


def rmsd_to_reference(trajectory, reference=None):
    """CA/C4' RMSD in Å to the input structure, or None without a reference."""
    matched = matched_reference(trajectory, reference)
    if matched is None:
        return None
    indices, target = matched
    return 10 * mdtraj.rmsd(trajectory.atom_slice(indices), target)


# ----------------------------------------------------------------------------- 3D viewers
_CHAIN_COLORS = ["#0078d4", "#e8590c", "#2f9e44", "#ae3ec9", "#f59f00", "#1098ad", "#e03131", "#5c7cfa"]
_PDB_CHAINS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
MAX_RENDERED_ATOMS = 300_000


def pdb_chain_ids(topology):
    """PDB chain IDs: BioEmu2's own when they are unique single characters, else A, B, C... by order."""
    ids = [chain.chain_id for chain in topology.chains]
    if all(isinstance(i, str) and len(i) == 1 for i in ids) and len(set(ids)) == len(ids):
        return ids
    return [_PDB_CHAINS[k % len(_PDB_CHAINS)] for k in range(len(ids))]


def pdb_text(trajectory, bfactors=None):
    """Multi-model PDB text with sequential atom serials and the chain IDs of pdb_chain_ids()."""
    atoms = list(trajectory.topology.atoms)
    chains = list(trajectory.topology.chains)
    serials = [atom.serial for atom in atoms]
    chain_ids = [chain.chain_id for chain in chains]
    letters = pdb_chain_ids(trajectory.topology)
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "frames.pdb"
        try:
            for serial, atom in enumerate(atoms, start=1):
                atom.serial = serial
            for letter, chain in zip(letters, chains, strict=True):
                chain.chain_id = letter
            trajectory.save_pdb(str(path), bfactors=bfactors)
        finally:
            for serial, atom in zip(serials, atoms, strict=True):
                atom.serial = serial
            for chain_id, chain in zip(chain_ids, chains, strict=True):
                chain.chain_id = chain_id
        return path.read_text()


def rmsf_bfactors(trajectory):
    """Per-atom B-factor column holding the CA/C4' RMSF (Å) of each atom's residue."""
    indices = representative_indices(trajectory)
    values = np.zeros(trajectory.n_atoms)
    if trajectory.n_frames > 1 and len(indices) >= 3:
        rmsf = rmsf_per_residue(trajectory, indices)
        by_residue = {trajectory.topology.atom(i).residue.index: v for i, v in zip(indices, rmsf)}
        for atom in trajectory.topology.atoms:
            values[atom.index] = by_residue.get(atom.residue.index, 0.0)
    return np.clip(values, 0, 99.0)


def _style(view, trajectory, color, viewer=None, max_b=None):
    kwargs = {} if viewer is None else {"viewer": viewer}
    if color == "chain":
        for index, letter in enumerate(pdb_chain_ids(trajectory.topology)):
            view.setStyle({"chain": letter},
                          {"cartoon": {"color": _CHAIN_COLORS[index % len(_CHAIN_COLORS)]}}, **kwargs)
    elif color == "secondary structure":
        view.setStyle({}, {"cartoon": {"colorscheme": "ssPyMol"}}, **kwargs)
    elif color == "flexibility (RMSF)":
        view.setStyle({}, {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb",
                                                       "min": max_b or 5.0, "max": 0.0}}}, **kwargs)
    else:
        view.setStyle({}, {"cartoon": {"color": "spectrum"}}, **kwargs)
    hetero = sorted({r.name for r in trajectory.topology.residues if not (r.is_protein or r.is_nucleic)})
    if hetero:
        view.addStyle({"resn": hetero}, {"stick": {"radius": 0.2}}, **kwargs)
    nucleic = sorted({r.name for r in trajectory.topology.residues if r.is_nucleic})
    if nucleic:
        view.addStyle({"resn": nucleic}, {"stick": {"radius": 0.12}}, **kwargs)


def show_ensemble(trajectory, *, frame_ids=None, layout="animation", color="spectrum", max_frames=20,
                  frame=0, interval=600, width=800, height=520):
    """Interactive py3Dmol view of a trajectory: animation, overlay, grid or single frame.

    ``frame_ids`` are the run frame numbers of the given frames (default 0, 1, ...); grid
    labels and the ``frame`` of the single layout use them.
    """
    frame_ids = np.arange(trajectory.n_frames) if frame_ids is None else np.asarray(frame_ids)
    trajectory = trajectory[:]
    fit = representative_indices(trajectory)
    trajectory.superpose(trajectory, frame=0, atom_indices=fit if len(fit) >= 3 else None)
    bfactors = rmsf_bfactors(trajectory) if color == "flexibility (RMSF)" else None
    max_b = float(np.percentile(bfactors, 98)) if bfactors is not None and bfactors.any() else None
    if layout == "single":
        frame = int(frame_ids[frame]) if frame < 0 else frame
        position = np.flatnonzero(frame_ids == frame)
        if len(position) == 0:
            raise IndexError(f"Frame {frame} is not among the selected frames (it failed the physicality "
                             "check or is out of range); choose another or show all frames.")
        subset = trajectory[int(position[0])]
        b = None if bfactors is None else bfactors[None]
        view = py3Dmol.view(width=width, height=height)
        view.addModel(pdb_text(subset, b), "pdb")
    else:
        # Every frame is embedded in the notebook, so cap the atoms rendered to keep it responsive.
        count = min(trajectory.n_frames, max_frames, max(2, MAX_RENDERED_ATOMS // trajectory.n_atoms))
        if count < min(trajectory.n_frames, max_frames):
            print(f"Showing {count} frames to keep the viewer responsive ({trajectory.n_atoms} atoms each).")
        frames = np.linspace(0, trajectory.n_frames - 1, count, dtype=int)
        subset = trajectory[frames]
        b = None if bfactors is None else np.repeat(bfactors[None], subset.n_frames, axis=0)
        if layout == "grid":
            columns = min(4, subset.n_frames)
            rows = math.ceil(subset.n_frames / columns)
            view = py3Dmol.view(width=max(width, 250 * columns), height=250 * rows,
                                viewergrid=(rows, columns), linked=True)
            for index in range(subset.n_frames):
                cell = divmod(index, columns)
                view.addModel(pdb_text(subset[index], None if b is None else b[index:index + 1]),
                              "pdb", viewer=cell)
                _style(view, subset, color, viewer=cell, max_b=max_b)
                view.addLabel(f"frame {frame_ids[frames[index]]}", {"position": {"x": 0, "y": 0, "z": 0},
                              "useScreen": True, "fontSize": 11, "backgroundOpacity": 0.4}, viewer=cell)
                view.zoomTo(viewer=cell)
            return view
        view = py3Dmol.view(width=width, height=height)
        if layout == "overlay":
            view.addModels(pdb_text(subset, b), "pdb")
        else:
            view.addModelsAsFrames(pdb_text(subset, b), "pdb")
            view.animate({"loop": "forward", "reps": 0, "interval": interval})
    _style(view, subset, color, max_b=max_b)
    view.zoomTo()
    return view


# ----------------------------------------------------------------------------- analysis
def rmsf_per_residue(trajectory, indices, fit=None):
    """RMSF (Å) of the ``indices`` atoms after superposing on the average of the ``fit`` atoms."""
    fit = indices if fit is None else fit
    traj = trajectory[:]
    traj.superpose(traj, frame=0, atom_indices=fit)
    average = mdtraj.Trajectory(traj.xyz.mean(axis=0, keepdims=True), traj.topology)
    traj.superpose(average, atom_indices=fit)
    sub = traj.xyz[:, indices]
    return 10 * np.sqrt(((sub - sub.mean(axis=0)) ** 2).sum(axis=-1).mean(axis=0))


def pairwise_rmsd(trajectory, indices, max_frames=300):
    frames = np.linspace(0, trajectory.n_frames - 1, min(trajectory.n_frames, max_frames), dtype=int)
    sub = trajectory[frames].atom_slice(indices)
    return frames, np.stack([10 * mdtraj.rmsd(sub, sub, frame=i) for i in range(sub.n_frames)])


def pca_projection(trajectory, indices):
    """First two principal components (Å) of superposed coordinates and their variance fractions."""
    sub = trajectory.atom_slice(indices)
    sub.superpose(sub, frame=0)
    flat = sub.xyz.reshape(sub.n_frames, -1) * 10
    centered = flat - flat.mean(axis=0)
    _, singular, vt = np.linalg.svd(centered, full_matrices=False)
    variance = singular**2 / max(float((singular**2).sum()), 1e-12)
    return centered @ vt[:2].T, variance[:2]


def residue_labels(trajectory, indices, author=None):
    """Tick labels for residues: author numbering when known, else chain and sequence position."""
    labels = []
    for i in indices:
        residue = trajectory.topology.atom(i).residue
        key = (residue.chain.chain_id, residue.resSeq)
        labels.append(author[key].replace(":", "") if author and key in author
                      else f"{key[0]}{key[1]}")
    return labels


def chain_boundaries(trajectory, indices):
    chains = [trajectory.topology.atom(i).residue.chain.index for i in indices]
    return [k for k in range(1, len(chains)) if chains[k] != chains[k - 1]]


# ----------------------------------------------------------------------------- report the stack
from importlib.metadata import version as _version
from bioemu2.biomoa.ops.triangle_multiplication import HAS_CUEQ

rows = [["BioEmu2", _version("bioemu2")], ["PyTorch", f"{torch.__version__} (CUDA {torch.version.cuda})"],
        ["Device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"],
        ["cuEquivariance kernels", "available" if HAS_CUEQ else "missing"],
        ["Triton", "available" if bioemu2.HAS_TRITON else "missing"],
        ["MDTraj / py3Dmol", f"{mdtraj.__version__} / {py3Dmol.__version__}"]]
show_table(rows, ["component", "status"])
if not torch.cuda.is_available():
    print("Warning: PyTorch cannot see a GPU. BioEmu2 will be extremely slow; see section 11.")
if not HAS_CUEQ:
    print("Warning: cuEquivariance is missing, so the default trunk kernels will fail for systems over "
          f"100 tokens. Choose kernels = 'pure PyTorch' in section 2, or: {CUDA_REMEDY}")
print("BioEmu2 is ready. Continue with section 2.")

# 2. Load the model

Downloads the BioEmu2 checkpoint (about 2 GB) from [Hugging Face](https://huggingface.co/microsoft/bioemu2) on first use. It is cached for the session, or on Google Drive if you enabled that in 1.3.

* **precision**: `bf16-mixed` (the default) runs the trunk in bfloat16 for speed and lower memory; `32-true` runs everything in float32. Denoising always runs in float32. GPUs older than Ampere (T4, V100) emulate bfloat16, so the speed gain there is small but the memory saving remains.
* **matmul_precision**: `high` lets Ampere and newer GPUs (A100, L4, H100) use TF32 tensor cores for denoising, several times faster with slightly lower precision; it has no effect on a T4.
* **compile_denoiser**: compiles the denoiser with `torch.compile`. The first batch takes longer, so it pays off only when a run has many batches.
* **kernels**: if a system runs out of memory while computing trunk embeddings even with `bf16-mixed`, switch to the memory-saving kernels. If the cuEquivariance kernels fail on this GPU, `pure PyTorch` avoids them. Both change results very slightly.

In [ ]:
#@title 2. Load the BioEmu2 model
checkpoint = "bioemu2-preview"  #@param ["bioemu2-preview"]
#@markdown Leave `local_checkpoint` empty to download from Hugging Face. To use your own copy, give the path of `checkpoint.ckpt`; its `config.yaml` must be in the same folder.
local_checkpoint = ""  #@param {type:"string"}
precision = "bf16-mixed"  #@param ["bf16-mixed", "32-true"]
matmul_precision = "high"  #@param ["high", "highest", "medium"]
compile_denoiser = False  #@param {type:"boolean"}
#@markdown **Trunk kernels.** *memory-saving* fits larger systems on one GPU at some cost in speed; *pure PyTorch* avoids the cuEquivariance kernels entirely (a fallback if they fail on this GPU, slower and more memory-hungry).
kernels = "checkpoint default"  #@param ["checkpoint default", "memory-saving (large systems)", "pure PyTorch (no cuEquivariance)"]
#@markdown Tile size for the memory-saving kernels; smaller tiles use less memory but run slower.
tile_size = 256  #@param [64, 128, 256, 512] {type:"raw"}

from omegaconf import OmegaConf

from bioemu2.checkpoints import get_checkpoint_files, resolve_checkpoint_files

if "require" not in globals():
    raise RuntimeError("Run cell 1.4 first.")

# Release the previous model completely, including sessions and embeddings that refer to it.
MODEL = None
drop_sessions()
EMBEDDINGS.clear()
free_gpu_memory()
start = time.time()
if local_checkpoint.strip():
    # Keep the path as given: load_model looks for config.yaml next to it, also when it is a link.
    checkpoint_path = Path(local_checkpoint.strip()).expanduser().absolute()
    if not checkpoint_path.is_file() or not (checkpoint_path.parent / "config.yaml").is_file():
        raise FileNotFoundError(f"Need {checkpoint_path} and config.yaml in the same folder.")
    files = resolve_checkpoint_files(checkpoint_path)
else:
    print(f"Fetching checkpoint '{checkpoint}' (downloaded once, then cached)...")
    files = get_checkpoint_files(checkpoint, cache_dir=HF_CACHE_DIR)
    checkpoint_path = files.checkpoint_path
config = OmegaConf.load(files.config_path)
if not kernels.startswith("checkpoint"):
    # Kernel variants are set in the checkpoint's config.yaml: write an edited copy next to a
    # link to the original weights. Links need a local disk, not Google Drive.
    variant_dir = Path("/content/bioemu2_kernels" if Path("/content").is_dir()
                       else Path.home() / "bioemu2_kernels") / slug(kernels.split(" ")[0])
    variant_dir.mkdir(parents=True, exist_ok=True)
    link = variant_dir / "checkpoint.ckpt"
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(files.checkpoint_path.resolve())
    if kernels.startswith("memory"):
        OmegaConf.update(config, "score_model.sequence_embedder.trimul_variant", "cueq_tiled")
        OmegaConf.update(config, "score_model.sequence_embedder.opm_variant", "triton_memory_saving")
        OmegaConf.update(config, "score_model.sequence_embedder.tiling_cfg.trimul_tile_size", tile_size,
                         merge=True)
    else:
        OmegaConf.update(config, "score_model.sequence_embedder.trimul_variant", "full")
        OmegaConf.update(config, "score_model.sequence_embedder.opm_variant", "vanilla")
    OmegaConf.save(config, variant_dir / "config.yaml")
    checkpoint_path = link

try:
    MODEL = load_model(checkpoint_path=checkpoint_path, precision=precision,
                       matmul_precision=matmul_precision if torch.cuda.is_available() else None,
                       compile_denoiser=compile_denoiser)
except RuntimeError as error:
    if "not supported by the installed PyTorch" in str(error):
        raise RuntimeError(f"{error}\n\n{CUDA_REMEDY}") from error
    raise

embedder = OmegaConf.select(config, "score_model.sequence_embedder") or {}
parameters = sum(p.numel() for p in MODEL.parameters())
MODEL_INFO = {"checkpoint": local_checkpoint.strip() or checkpoint, "precision": precision,
              "matmul_precision": matmul_precision, "compile_denoiser": compile_denoiser,
              "kernels": kernels}
show_table([["checkpoint", local_checkpoint.strip() or checkpoint],
            ["parameters", f"{parameters / 1e6:.0f} M"],
            ["device", str(MODEL.device)],
            ["precision", precision],
            ["trunk kernels", f"{embedder.get('trimul_variant', 'cueq')} triangle multiplication, "
                              f"{embedder.get('opm_variant', 'vanilla')} outer product"],
            ["supports propagation", MODEL.supports_propagation],
            ["GPU memory", gpu_memory_report()],
            ["load time", f"{time.time() - start:.0f} s"]], ["setting", "value"])

# 3. Define the molecular system

Choose **one** input type and fill in the matching fields; the other fields are ignored.

| Input type | Use it for | Notes |
|---|---|---|
| **PDB ID** | Any RCSB entry | Loads biological assembly 1 by default, with ligands except water. Unresolved residues are included from the entry's sequence records. |
| **Sequences** | Proteins, RNA, DNA, ligands by CCD code or SMILES | Separate chains with `:` (ColabFold style), e.g. `MKV...:GSHM...`; spaces and line breaks inside a sequence are ignored. Repeat a sequence for a homo-oligomer. |
| **Structure file** | Your own mmCIF or PDB file | mmCIF files with entity records (RCSB, PDB-REDO, AlphaFold DB) give the most faithful system. For PDB files the sequence is read from the resolved residues. |
| **BioEmu2 system YAML** | Full control: modified residues, glycans, covalent bonds, contacts | See the [system guide](https://github.com/microsoft/bioemu2/blob/main/docs/system.md). The `system.yaml` files this notebook and the CLI write can be loaded here. |
| **Boltz input YAML** | Re-using Boltz inputs | Converted with `convert_boltz`; unsupported fields are reported. |
| **Built-in example** | Trying things out | Systems shipped with BioEmu2. |

Model cost grows with the number of **tokens**: one per standard polymer residue, plus one per heavy atom of each ligand or modified residue.

**Ligands in PDB entries.** Bound ligands, ions and crystallisation additives are kept unless you exclude them. The default entry 1AKE contains the inhibitor Ap5A (`AP5`), which locks adenylate kinase in its closed state; for the free (apo) enzyme, untick `include_ligands` or choose the built-in example `ake`.

**Disulfide bonds** are not read from structure files, but `declare_disulfides` detects them from the coordinates (S–S distance below 2.5 Å) and declares them. Steering samplers keep declared disulfides at bond length and push undeclared ones apart. For sequence input, declare disulfides in a system YAML (`bonds:`).

In [ ]:
#@title 3.1 Define the system
input_type = "PDB ID"  #@param ["PDB ID", "Sequences", "Structure file (mmCIF or PDB)", "BioEmu2 system YAML", "Boltz input YAML", "Built-in example"]
#@markdown Name for the output folder; leave empty to derive it from the input. Re-using a name lets 5.2 resume interrupted runs of that job.
job_name = ""  #@param {type:"string"}
#@markdown ---
#@markdown **PDB ID**
pdb_id = "1AKE"  #@param {type:"string"}
assembly = "1"  #@param ["1", "2", "3", "asymmetric unit"] {allow-input: true}
#@markdown Author chain IDs to keep, e.g. `A` or `A, B`; empty keeps all chains.
chains = ""  #@param {type:"string"}
#@markdown ---
#@markdown **Sequences**: separate chains with `:`. Ligands: comma-separated CCD codes (e.g. `ATP, MG`) and/or SMILES.
protein_sequences = ""  #@param {type:"string"}
rna_sequences = ""  #@param {type:"string"}
dna_sequences = ""  #@param {type:"string"}
ligand_ccd_codes = ""  #@param {type:"string"}
ligand_smiles = ""  #@param {type:"string"}
#@markdown ---
#@markdown **Structure file / YAML**: path to the file, or leave empty to open an upload dialog.
input_file = ""  #@param {type:"string"}
#@markdown ---
#@markdown **Built-in example**
example = "ake: E. coli adenylate kinase"  #@param ["ake: E. coli adenylate kinase", "chignolin: 10-residue mini-protein", "myoglobin_101m: myoglobin with heme, sulfate and NBN (with a contact)", "meteorin_like_7tlw: NTR domain with two disulfides", "shp2_3tkz: SH2 domain with two peptides (with a contact)"]
#@markdown ---
#@markdown **Ligands** (PDB ID and structure files): keep ligands, optionally excluding CCD codes such as crystallisation additives (`SO4, GOL, EDO`).
include_ligands = True  #@param {type:"boolean"}
exclude_ligands = ""  #@param {type:"string"}
#@markdown **Disulfides** (PDB ID and structure files): detect and declare S–S bonds.
declare_disulfides = True  #@param {type:"boolean"}
show_input_structure = True  #@param {type:"boolean"}

import gzip

import requests
import yaml

if "require" not in globals():
    raise RuntimeError("Run cell 1.4 first.")

RCSB_ASSEMBLY = "https://files.rcsb.org/download/{pdb}-assembly{assembly}.cif.gz"
RCSB_ENTRY = "https://files.rcsb.org/download/{pdb}.cif"

excluded = [code.upper() for code in split_list(exclude_ligands)]
ligand_kwargs = {"include_ligands": include_ligands}
if include_ligands and excluded:
    ligand_kwargs["exclude_ligands"] = excluded

new_contacts = None
reference, reference_author, reference_file, cysteines, derived_name = None, None, None, [], "custom"
input_dir = UPLOAD_DIR / "inputs"
input_dir.mkdir(parents=True, exist_ok=True)

if input_type == "PDB ID":
    pdb = pdb_id.strip().upper()
    if not re.fullmatch(r"[0-9][A-Z0-9]{3}", pdb):
        raise ValueError(f"{pdb_id!r} is not a four-character PDB ID.")
    assembly_id = None if assembly.startswith("asym") else assembly.strip()
    author_chains = [c for c in re.split(r"[\s,;]+", chains.strip()) if c] or None
    try:
        new_system = System.from_pdb_id(pdb, assembly_id=assembly_id, auth_asym_ids=author_chains,
                                        **ligand_kwargs)
    except requests.HTTPError as error:
        raise RuntimeError(
            f"RCSB has no {'entry' if assembly_id is None else f'assembly {assembly_id}'} for {pdb} "
            f"({error}). Check the PDB ID, or choose assembly 1 or 'asymmetric unit'."
        ) from error
    url = (RCSB_ENTRY.format(pdb=pdb) if assembly_id is None
           else RCSB_ASSEMBLY.format(pdb=pdb, assembly=assembly_id))
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    content = gzip.decompress(response.content) if url.endswith(".gz") else response.content
    reference_file = input_dir / (f"{pdb}.cif" if assembly_id is None else f"{pdb}-assembly{assembly_id}.cif")
    reference_file.write_bytes(content)
    reference, reference_author = reference_from_cif(reference_file)
    cysteines = cysteines_from_cif(reference_file, new_system.chain_ids)
    derived_name = pdb + (f"_{'_'.join(author_chains)}" if author_chains else "")

elif input_type == "Sequences":
    def _sequences(text, alphabet, kind):
        # Chains are separated by ':' (also ';' or '/'); whitespace inside a sequence is ignored.
        sequences = ["".join(chunk.split()).upper() for chunk in re.split(r"[:;/]", text) if chunk.strip()]
        for sequence in sequences:
            bad = sorted(set(sequence) - set(alphabet))
            if bad:
                hint = " Remove residue numbers." if any(ch.isdigit() for ch in bad) else ""
                raise ValueError(f"{kind} sequence contains invalid characters {bad}: {sequence[:30]}...{hint}")
        return sequences

    proteins = _sequences(protein_sequences, "ACDEFGHIKLMNPQRSTVWYX", "Protein")
    rnas = _sequences(rna_sequences, "ACGUN", "RNA")
    dnas = _sequences(dna_sequences, "ACGTN", "DNA")
    ccd = [code.upper() for code in re.split(r"[\s,;]+", ligand_ccd_codes.strip()) if code]
    smiles = [s for s in re.split(r"[\s,]+", ligand_smiles.strip()) if s]
    if not (proteins or rnas or dnas or ccd or smiles):
        raise ValueError("Enter at least one sequence or ligand.")
    new_system = System.from_description(proteins=proteins, rna=rnas, dna=dnas,
                                         ligand_ccd_codes=ccd, ligand_smiles=smiles)
    # Named after the content, so re-running with the same input finds the same job folder.
    derived_name = "sequence_" + hashlib.sha1(str(new_system).encode()).hexdigest()[:8]
    if len(proteins) > 1:
        print(f"{len(proteins)} protein chains of lengths {', '.join(str(len(p)) for p in proteins)}.")
    if any(p.count("C") >= 2 for p in proteins):
        print("Note: disulfides cannot be detected from sequences. If the protein has any, declare them "
              "in a system YAML (bonds) when using steering samplers.")

elif input_type.startswith("Structure file"):
    path = get_files(input_file, "Upload an mmCIF (.cif) or PDB (.pdb) file:")[0]
    if path.name.lower().endswith(".gz"):
        unzipped = input_dir / Path(path.name[:-3]).name
        unzipped.write_bytes(gzip.decompress(path.read_bytes()))
        path = unzipped
    derived_name = path.stem
    reference_file = path
    if path.suffix.lower() in (".cif", ".mmcif"):
        try:
            new_system = System.from_cif(path, **ligand_kwargs)
            reference, reference_author = reference_from_cif(path)
            cysteines = cysteines_from_cif(path, new_system.chain_ids)
        except Exception as error:
            print(f"The mmCIF file lacks the entity records BioEmu2 needs ({error}).\n"
                  "Reading the sequence from the resolved residues instead.")
            new_system, reference, reference_author, cysteines = structure_file_to_system(
                path, include_ligands=include_ligands, exclude_ligands=excluded)
    else:
        new_system, reference, reference_author, cysteines = structure_file_to_system(
            path, include_ligands=include_ligands, exclude_ligands=excluded)

elif input_type == "BioEmu2 system YAML":
    path = get_files(input_file, "Upload a BioEmu2 system YAML file:")[0]
    data = yaml.safe_load(path.read_text())
    path_to_load = path
    if isinstance(data, dict) and "mode" in data:
        # The bioemu2 CLI records the task mode in its system.yaml; the task is chosen in section 5.
        print(f"Ignoring 'mode: {data.pop('mode')}' from the file; choose the task in section 5.")
        path_to_load = unique_dir(input_dir / f"{path.stem}_system.yaml")
        path_to_load.write_text(yaml.safe_dump(data, sort_keys=False))
    new_system, new_contacts = load_system_and_contacts(path_to_load)
    derived_name = path.stem

elif input_type == "Boltz input YAML":
    from bioemu2 import convert_boltz

    path = get_files(input_file, "Upload a Boltz input YAML file:")[0]
    converted = unique_dir(input_dir / f"{path.stem}_bioemu2.yaml")
    convert_boltz(path, converted)
    new_system, new_contacts = load_system_and_contacts(converted)
    print(f"Converted to {converted}")
    derived_name = path.stem

else:
    name = example.split(":")[0]
    new_system, new_contacts = load_system_and_contacts(EXAMPLES_DIR / "systems" / f"{name}.yaml")
    derived_name = name

disulfides = []
if declare_disulfides and cysteines:
    new_system, disulfides = add_disulfides(new_system, cysteines)

# Commit the new system and forget results that belonged to the previous one.
SYSTEM_TOKENS = None
SYSTEM, CONTACTS = new_system, new_contacts
REFERENCE, REFERENCE_AUTHOR, REFERENCE_FILE = reference, reference_author, reference_file
reset_downstream()
JOB_NAME = slug(job_name) if job_name.strip() else slug(derived_name)
JOB_DIR = OUTPUT_ROOT / JOB_NAME
(JOB_DIR / "input").mkdir(parents=True, exist_ok=True)
write_system_yaml(JOB_DIR / "input" / "system.yaml", SYSTEM, CONTACTS)
if REFERENCE_FILE is not None:
    destination = JOB_DIR / "input" / REFERENCE_FILE.name
    if not (destination.exists() and destination.samefile(REFERENCE_FILE)):
        shutil.copy(REFERENCE_FILE, destination)

SYSTEM_TOKENS = count_tokens(SYSTEM)
n_atoms = SYSTEM.topology.n_atoms
show_table(system_entities_table(SYSTEM), ["chain", "author chain", "type", "size", "sequence / identity"])
estimate = max_batch_estimate(SYSTEM_TOKENS)
print(f"Job '{JOB_NAME}': {len(SYSTEM.chain_ids)} chains, {SYSTEM_TOKENS} tokens, {n_atoms} heavy atoms.")
if REFERENCE:
    print(f"Reference structure: {REFERENCE_FILE.name} ({len(REFERENCE)} resolved residues), used for RMSD.")
if disulfides:
    print(f"Declared {len(disulfides)} disulfide bond(s) found in the input structure: {', '.join(disulfides)}.")
if SYSTEM.bonds and not disulfides:
    print(f"The system declares {len(SYSTEM.bonds)} extra covalent bond(s).")
if CONTACTS:
    print(f"Contacts from the file: {len(CONTACTS)} (see 3.2).")
if torch.cuda.is_available() and estimate < 1:
    print("Warning: this system is likely too large for this GPU's memory. Use a GPU with more "
          "memory, fewer chains, the memory-saving kernels (section 2) or fewer MSA rows (section 4).")
print(f"Output folder: {JOB_DIR}")

if show_input_structure and REFERENCE_FILE is not None:
    view = py3Dmol.view(width=700, height=420)
    view.addModel(REFERENCE_FILE.read_text(), "cif" if REFERENCE_FILE.suffix.lower() in (".cif", ".mmcif") else "pdb")
    view.setStyle({}, {"cartoon": {"color": "spectrum"}})
    view.addStyle({"hetflag": True}, {"stick": {"radius": 0.2}})
    view.setStyle({"resn": ["HOH", "WAT"]}, {})
    view.zoomTo()
    view.show()

In [ ]:
#@title 3.2 Contact conditioning (optional)
#@markdown Ask the model to bring pairs of tokens within a maximum distance. Separate contacts with `;`. Each endpoint is `chain:residue` (1-based residue number), `chain:ATOM` for a ligand atom, or `chain:residue:ATOM`; append `@distance` in Å (4–20, default 6).
#@markdown Example: `A:96-C:FE@6; A:35-B:4`. Use the **chain** column from 3.1 (canonical chain IDs such as `A` or `A-2`, not author chain IDs) and count residues from 1 along the full sequence shown there (not PDB author numbering). On standard residues only `CA` (protein) or `C1'` (nucleotide) can be named; SMILES ligands have atom names `C1, C2, O1, ...` in canonical order.
contacts = ""  #@param {type:"string"}
#@markdown Contacts loaded from a YAML file in 3.1 are kept unless you enter new ones or tick `clear_contacts`.
clear_contacts = False  #@param {type:"boolean"}

require("SYSTEM")


def parse_token(text):
    parts = [p.strip() for p in text.strip().split(":")]
    if len(parts) == 2 and parts[1].isdigit():
        return ContactToken(asym_id=parts[0], res_id=int(parts[1]))
    if len(parts) == 2:
        return ContactToken(asym_id=parts[0], atom_name=parts[1])
    if len(parts) == 3 and parts[1].isdigit():
        return ContactToken(asym_id=parts[0], res_id=int(parts[1]), atom_name=parts[2])
    raise ValueError(f"Cannot parse contact endpoint {text!r}; use chain:residue, chain:ATOM or "
                     f"chain:residue:ATOM. Chains: {', '.join(SYSTEM.chain_ids)}")


def parse_contacts(text):
    parsed = []
    for entry in [e for e in text.split(";") if e.strip()]:
        pair, _, distance = entry.partition("@")
        # Chain IDs may contain '-' (e.g. A-2), but the part after the first ':' may not.
        match = re.fullmatch(r"\s*([^:]+:[^-]+?)\s*-\s*(\S.*?)\s*", pair)
        if not match:
            raise ValueError(f"Contact {entry.strip()!r} needs two endpoints, e.g. A:10-B:25@8. "
                             f"Chains: {', '.join(SYSTEM.chain_ids)}")
        left, right = match.groups()
        kwargs = {"max_distance": float(distance)} if distance.strip() else {}
        parsed.append(Contact(token1=parse_token(left), token2=parse_token(right), **kwargs))
    return parsed


from bioemu2.structure import AllAtomStructure

previous = repr(CONTACTS)
new_contacts = None if clear_contacts else (parse_contacts(contacts) if contacts.strip() else CONTACTS)
if new_contacts:
    SYSTEM.validate_contacts(new_contacts)
    # Building the structure also checks atom names, which validate_contacts does not.
    AllAtomStructure.from_system(SYSTEM, mode="structure", contacts=new_contacts)
CONTACTS = new_contacts
if repr(CONTACTS) != previous:
    drop_sessions()
    EMBEDDINGS.clear()
    free_gpu_memory()
    write_system_yaml(JOB_DIR / "input" / "system.yaml", SYSTEM, CONTACTS)

if CONTACTS:
    def _endpoint(token):
        if token.atom_name and "res_id" not in token.model_fields_set:
            return f"{token.asym_id}:{token.atom_name}"
        return f"{token.asym_id}:{token.res_id}" + (f":{token.atom_name}" if token.atom_name else "")

    show_table([[i + 1, _endpoint(c.token1), _endpoint(c.token2), f"{c.max_distance:g} Å"]
                for i, c in enumerate(CONTACTS)], ["#", "token 1", "token 2", "max distance"])
else:
    print("No contact conditioning.")

# 4. Multiple sequence alignment (MSA)

BioEmu2 uses MSAs of the **protein** chains as evolutionary input (RNA, DNA and ligands need none).

* **ColabFold MMseqs2 server**: searches UniRef and, optionally, environmental databases; heteromers also get paired alignments. Results are cached, so re-running is free.
* **Upload A3M file(s)**: alignments from your own pipeline or an earlier run (each run saves its alignments in the job's `msa/` folder). Upload one file per distinct protein sequence; the first record must be the query, and files are matched to chains by sequence. Uploaded heteromer alignments are unpaired, and their depth is cut to the shallowest chain.
* **Single sequence**: no evolutionary information. Fast, but predictions are usually much worse.

`max_msa_rows` caps the number of alignment rows the model receives, to save GPU memory and time on large systems; for heteromers the budget is split between paired and unpaired rows. The model uses at most the first 4,092 rows anyway.

The ColabFold client has no overall time limit. If a search is still running after 15 minutes, interrupt the cell (■) and try again later.

In [ ]:
#@title 4. Build the MSA
msa_source = "ColabFold MMseqs2 server"  #@param ["ColabFold MMseqs2 server", "Upload A3M file(s)", "Single sequence (no MSA)"]
search_environmental_databases = True  #@param {type:"boolean"}
#@markdown Heteromer pairing: *auto* uses `complete` for three or more distinct protein sequences and `greedy` otherwise.
pairing_strategy = "auto"  #@param ["auto", "greedy", "complete"]
#@markdown `0` keeps every row.
max_msa_rows = 0  #@param {type:"integer"}
#@markdown For uploads: comma-separated paths, or leave empty to open an upload dialog.
a3m_files = ""  #@param {type:"string"}
show_coverage_plot = True  #@param {type:"boolean"}

require("SYSTEM")


def _protein_chains(system):
    return [(chain, entity.sequence) for entity in system.entities if isinstance(entity, Protein)
            for chain in entity.asym_ids]


def _a3m_records(text):
    """(header, sequence) records of an A3M string, joining wrapped sequence lines."""
    records, header, lines = [], None, []
    for line in text.splitlines():
        if line.startswith(">"):
            if header is not None:
                records.append((header, "".join(lines)))
            header, lines = line[1:], []
        elif line.strip() and not line.startswith("#"):
            lines.append(line.strip())
    if header is not None:
        records.append((header, "".join(lines)))
    return records


def _clean_a3m(path, folder):
    """Write a normalised copy: one line per sequence, no secondary-structure records or '.'/'*'."""
    text = path.read_text()
    if text.startswith("#") and "," in text.splitlines()[0]:
        raise ValueError(f"{path.name} is a ColabFold complex A3M; upload one A3M per chain instead.")
    records = [(h, re.sub(r"[.*\s]", "", seq)) for h, seq in _a3m_records(text)
               if not h.startswith(("ss_pred", "ss_conf", "ss_dssp"))]
    if not records:
        raise ValueError(f"{path.name} contains no A3M records.")
    clean = unique_dir(folder / path.name)
    clean.write_text("".join(f">{h}\n{seq}\n" for h, seq in records))
    return clean, re.sub(r"[a-z\-]", "", records[0][1]).upper()


def _depth(a3m):
    return sum(line.startswith(">") for line in a3m.splitlines())


def _cap_msa(msa, rows):
    """Cap the rows the model receives (query + paired + unpaired rows, minus the repeated query)."""
    if msa.paired_a3m is None:
        return msa[:rows]
    paired_depth = min(_depth(a3m) for a3m in msa.paired_a3m.values())
    n_paired = max(1, min(paired_depth, (rows + 1) // 2))
    return MSA(unpaired_a3m=msa[:rows + 1 - n_paired].unpaired_a3m, paired_a3m=msa[:n_paired].paired_a3m)


chains_with_sequences = _protein_chains(SYSTEM)
start = time.time()
if not chains_with_sequences:
    msa = get_msa(SYSTEM)  # No protein chains: an empty MSA, no network access.
    print("The system has no protein chains, so no MSA is needed.")
elif msa_source.startswith("ColabFold"):
    distinct = len({sequence for _, sequence in chains_with_sequences})
    strategy = pairing_strategy if pairing_strategy != "auto" else ("complete" if distinct >= 3 else "greedy")
    print("Querying the ColabFold MMseqs2 server (usually 1–5 minutes; cached afterwards)...")
    try:
        msa = get_msa(SYSTEM, use_env=search_environmental_databases, pairing_strategy=strategy)
    except Exception as error:
        raise RuntimeError(
            f"The ColabFold MSA search failed: {error}\nThe public server may be busy or rate "
            "limiting this runtime. Wait a few minutes and retry, upload A3M files, or use "
            "single-sequence mode."
        ) from error
elif msa_source.startswith("Upload"):
    paths = get_files(a3m_files, "Upload one A3M file per distinct protein sequence:", multiple=True)
    folder = Path(tempfile.mkdtemp(dir=UPLOAD_DIR, prefix="a3m_"))
    cleaned = [_clean_a3m(path, folder) for path in paths]
    unique = {sequence for _, sequence in chains_with_sequences}
    if len(unique) == 1 and len(cleaned) == 1:
        source = cleaned[0][0]
    else:
        by_query = {}
        for clean, query in cleaned:
            by_query.setdefault(query, clean)
        source = {}
        for chain, sequence in chains_with_sequences:
            match = by_query.get(sequence.upper())
            if match is None:
                raise ValueError(f"No uploaded A3M has chain {chain}'s sequence as its first record.")
            source[chain] = match
    msa = get_msa(SYSTEM, source=source)
else:
    msa = MSA({chain: f">query\n{sequence}\n" for chain, sequence in chains_with_sequences})

if max_msa_rows > 0 and chains_with_sequences:
    msa = _cap_msa(msa, max_msa_rows)
MSA_OBJ, MSA_FOR = msa, system_key(SYSTEM)
drop_sessions()
EMBEDDINGS.clear()

if chains_with_sequences:
    msa_dir = JOB_DIR / "msa"
    msa_dir.mkdir(parents=True, exist_ok=True)
    rows, plots = [], []
    for chain, sequence in chains_with_sequences:
        text = MSA_OBJ.a3m(chain)
        (msa_dir / f"{chain}.a3m").write_text(text)
        records = _a3m_records(text)
        paired = _depth(MSA_OBJ.a3m(chain, paired=True)) if MSA_OBJ.paired_a3m is not None else 0
        model_rows = min(4092, len(records) + max(paired - 1, 0))
        rows.append([chain, len(sequence), len(records), paired or "–", model_rows])
        if chain == next(c for c, s in chains_with_sequences if s == sequence):
            aligned = [re.sub(r"[a-z.]", "", seq) for _, seq in records]
            plots.append((chain, sequence, aligned))
    show_table(rows, ["chain", "length", "unpaired rows", "paired rows", "rows used by the model"])
    print(f"MSA ready in {time.time() - start:.0f} s; A3M files saved to {msa_dir}")

    if show_coverage_plot:
        fig, axes = plt.subplots(1, len(plots), figsize=(min(16, 5 * len(plots) + 2), 3.4), squeeze=False)
        for ax, (chain, sequence, aligned) in zip(axes[0], plots):
            rows_ok = [row for row in aligned if len(row) == len(sequence)][:5000]
            if not rows_ok:
                ax.set_axis_off()
                continue
            matrix = np.array([list(row) for row in rows_ok])
            query = np.array(list(sequence.upper()))
            covered = matrix != "-"
            identity = (matrix == query).sum(axis=1) / np.maximum(covered.sum(axis=1), 1)
            order = np.argsort(-identity)
            image = np.where(covered, identity[:, None], np.nan)[order]
            ax.imshow(image, aspect="auto", cmap="rainbow_r", vmin=0, vmax=1, interpolation="nearest",
                      extent=(0.5, len(sequence) + 0.5, 0, len(rows_ok)))
            ax.plot(np.arange(1, len(sequence) + 1), covered.sum(axis=0), color="black", lw=1)
            ax.set_xlabel("residue")
            ax.set_ylabel("sequences")
            ax.set_title(f"chain {chain}: {len(rows_ok)} sequences")
        fig.suptitle("MSA coverage, coloured by identity to the query (black: sequences covering each residue)",
                     fontsize=10)
        fig.tight_layout()
        plt.show()

# 5. Generate structures

BioEmu2 has two task modes, which use the same denoising algorithm with different conditioning:

* **Structure prediction** (`mode="structure"`) gives accurate individual structures. With `edm`, `edm_gd`, `edm_fk`, `fk_resampling_only` and the example custom sampler it also sharpens the distribution with `step_scale=1.5`. The FKC samplers (`edm_fkc`, `fkc_smc`) have no step scale and never sharpen; with them, structure mode differs from equilibrium mode only in its conditioning.
* **Equilibrium sampling** (`mode="equilibrium"`) gives a diverse ensemble of conformations the system can adopt; it does not sharpen (`step_scale=1.0`).

**Samplers.** `edm` is plain, unsteered sampling. The steering presets nudge samples towards physically plausible geometry (fewer clashes and broken bonds) using the default physicality potentials, at extra cost:
`edm_gd` adds gradient guidance (a good first choice); `edm_fk` adds guidance plus Feynman–Kac resampling; `edm_fkc` uses Feynman–Kac correctors with guidance through the denoiser (most expensive in memory). Two guidance-free variants reweight samples without gradients: `fk_resampling_only` and `fkc_smc`. Resampling samplers treat each batch as one population, so use batches of 8 or more with them. `custom` uses the sampler defined in cell 5.1. See the [steering guide](https://github.com/microsoft/bioemu2/blob/main/docs/steering.md). Steering a system that contains ligands downloads the Boltz chemical-component library from Hugging Face on first use.

**Physicality.** The protein chains of every sample are checked for bad bonds, clashes and similar problems. Ligands, nucleic acids and protein–ligand contacts are not checked. All frames are kept, and frames that pass are also saved separately (in `passed/`).

Each run gets its own folder under the job folder with `samples.xtc` (all frames), `topology.cif`, `physicality.json`, `system.yaml` and `run_info.json`. Load them later with `mdtraj.load_xtc("samples.xtc", top="topology.cif")` after `import bioemu2`.

**Resuming.** With `resumable` ticked, finished batches are saved as they complete. If the runtime disconnects, run cells 1.1 to 4 again with the same inputs (same job name), then 5.2 with the same settings: it finds the unfinished run and continues it, reusing its seed and batch size. With Google Drive enabled in 1.3 this also works in a new session.

In [ ]:
#@title 5.1 (Optional) Custom sampler and steering potentials
#@markdown Edit the function below, run this cell, then choose `sampler = custom` in cell 5.2. Not used otherwise.
from bioemu2 import EDMSampler, FKCSampler, GuidedFKSampler
from bioemu2.denoising.steering.potentials import BondLengthPotential, get_default_potentials


def make_custom_sampler(mode, num_steps):
    """Return the sampler that cell 5.2 uses when sampler = "custom".

    mode is "structure" or "equilibrium". The example restrains bond lengths with
    gradient guidance and no resampling, the algorithm of the edm_gd preset.
    """
    step_scale = 1.5 if mode == "structure" else 1.0
    return GuidedFKSampler(
        num_steps=num_steps,
        step_scale=step_scale,
        potentials=[
            BondLengthPotential(sigma_tolerance=10.0, parameters={"guidance_weight": 0.05}),
        ],
        resampling_interval=None,
        num_gd_steps=20,
    )
    # Other options:
    #   EDMSampler(num_steps=num_steps, step_scale=step_scale, noise_scale=1.0)
    #   GuidedFKSampler(num_steps=num_steps, step_scale=step_scale, potentials=get_default_potentials())
    #   FKCSampler(num_steps=num_steps, weight_update="smc")  # no step_scale: never sharpens


print("Custom sampler defined:", type(make_custom_sampler("equilibrium", 200)).__name__)

In [ ]:
#@title 5.2 Generate structures
task = "Equilibrium ensemble"  #@param ["Equilibrium ensemble", "Structure prediction"]
num_samples = 16  #@param {type:"integer"}
#@markdown `auto` picks a batch size from the system size and GPU memory and halves it automatically if the GPU runs out of memory.
batch_size = "auto"  #@param ["auto", "1", "2", "4", "8", "16", "32", "64"] {allow-input: true}
sampler = "edm"  #@param ["edm", "edm_gd", "edm_fk", "edm_fkc", "fk_resampling_only", "fkc_smc", "custom"]
#@markdown Denoising steps (default 200). Fewer steps run faster but change and usually degrade the samples.
num_steps = 200  #@param {type:"integer"}
#@markdown Optional YAML with a top-level `potentials:` list that replaces the default steering potentials (steering samplers only); leave empty for the defaults.
potentials_yaml = ""  #@param {type:"string"}
#@markdown Random seed; `-1` draws a new one. Batch *i* uses `seed + i`.
seed = 0  #@param {type:"integer"}
physicality_check = "high_quality"  #@param ["high_quality", "high_quality_per_residue", "no_violations", "no_filter"]
#@markdown Name of the run folder; empty derives one from the task and sampler.
run_name = ""  #@param {type:"string"}
#@markdown Save finished batches as they complete, so a run interrupted by a disconnect can be resumed (see above).
resumable = True  #@param {type:"boolean"}

import inspect
import random

from bioemu2 import FKCSampler, GuidedFKSampler

require("MODEL", "SYSTEM", "MSA_OBJ")
mode = "structure" if task.startswith("Structure") else "equilibrium"
if num_samples < 1:
    raise ValueError("num_samples must be at least 1.")
if num_steps < 2:
    raise ValueError("num_steps must be at least 2.")
if batch_size.strip().lower() != "auto" and int(batch_size) < 1:
    raise ValueError("batch_size must be at least 1.")


def _sha256(data):
    return hashlib.sha256(data if isinstance(data, bytes) else data.encode()).hexdigest()


potentials, potentials_sha = None, None
if potentials_yaml.strip():
    if sampler in ("edm", "custom"):
        raise ValueError("potentials_yaml needs a steering sampler (not edm or custom).")
    from hydra.utils import instantiate
    from omegaconf import OmegaConf

    potentials_sha = _sha256(Path(potentials_yaml.strip()).read_bytes())
    config = OmegaConf.load(potentials_yaml.strip())
    if "potentials" not in config:
        raise ValueError(f"{potentials_yaml} must have a top-level 'potentials:' list.")
    potentials = tuple(instantiate(config.potentials))

custom_source = None
if sampler == "custom":
    if "make_custom_sampler" not in globals():
        raise RuntimeError("Run cell 5.1 first to define make_custom_sampler.")
    try:
        custom_source = inspect.getsource(make_custom_sampler)
    except (OSError, TypeError):
        code_object = make_custom_sampler.__code__
        custom_source = code_object.co_code.hex() + repr(code_object.co_consts)
    configured_sampler = make_custom_sampler(mode, num_steps)
elif sampler == "fk_resampling_only":
    configured_sampler = GuidedFKSampler(num_steps=num_steps, step_scale=1.5 if mode == "structure" else 1.0,
                                         num_gd_steps=0, **({"potentials": potentials} if potentials else {}))
elif sampler == "fkc_smc":
    configured_sampler = FKCSampler(num_steps=num_steps, weight_update="smc",
                                    **({"potentials": potentials} if potentials else {}))
else:
    configured_sampler = sampler_from_preset(sampler, mode=mode, num_steps=num_steps, potentials=potentials)
if mode == "structure" and isinstance(configured_sampler, FKCSampler):
    print("Note: FKC samplers have no step_scale, so structure-mode sharpening (step_scale=1.5) is not applied.")

# A run folder is resumed only if an interrupted run with the same requested settings and inputs
# left it. The seed and batch size it actually used are stored separately and reused.
name = slug(run_name) if run_name.strip() else f"{mode}_{sampler}"
match = {"mode": mode, "num_samples": num_samples, "sampler": sampler, "num_steps": num_steps,
         "seed": seed, "batch_size": batch_size.strip().lower(), "potentials_sha256": potentials_sha,
         "custom_sampler_sha256": _sha256(custom_source) if custom_source else None,
         "inputs": _sha256(str(SYSTEM) + repr(CONTACTS) + repr(sorted((c, MSA_OBJ.a3m(c)) for c in MSA_OBJ.chain_ids))
                           + repr(sorted((c, MSA_OBJ.a3m(c, paired=True)) for c in MSA_OBJ.chain_ids)
                                  if MSA_OBJ.paired_a3m is not None else "")
                           + json.dumps(MODEL_INFO, sort_keys=True))}
run_dir, resumed, unfinished = None, None, []
for index in range(1, 100_000):
    candidate = JOB_DIR / (name if index == 1 else f"{name}_{index}")
    if not candidate.exists():
        run_dir = candidate
        break
    pending_path = candidate / "pending.json"
    if (candidate / "samples.xtc").exists() or not pending_path.exists():
        continue
    stored = json.loads(pending_path.read_text())
    if resumable and stored.get("match") == match:
        run_dir, resumed = candidate, stored
        break
    unfinished.append((candidate.name, stored))
for folder, stored in unfinished:
    differing = sorted(k for k in match if stored.get("match", {}).get(k) != match[k])
    print(f"Note: {folder} holds an unfinished run with different settings ({', '.join(differing)}); "
          "starting a new run.")
if resumed:
    seed, size = resumed["seed"], resumed["batch_size"]
    done = len(list((run_dir / "partial").glob("*.done"))) if (run_dir / "partial").exists() else 0
    print(f"Resuming the unfinished run in {run_dir.name} (seed {seed}, batch size {size}, {done} finished batches).")
else:
    if seed < 0:
        seed = random.randrange(2**31 - 1)
    size = auto_batch_size(num_samples, sampler) if batch_size.strip().lower() == "auto" else int(batch_size)
if sampler in ("edm_fk", "edm_fkc", "fk_resampling_only", "fkc_smc") and min(size, num_samples) < 8:
    print(f"Note: {sampler} resamples within each batch; a population of {min(size, num_samples)} is "
          "small and can reduce diversity. Use 8 or more samples per batch if memory allows.")
partial_dir = run_dir / "partial" if resumable else None
run_dir.mkdir(parents=True, exist_ok=True)
pending = {"match": match, "seed": seed, "batch_size": size}
(run_dir / "pending.json").write_text(json.dumps(pending))

embeddings = embeddings_for(mode)
session = EmuSession(SYSTEM, model=MODEL, mode=mode, msa=MSA_OBJ, contacts=CONTACTS, embeddings=embeddings)
print(f"Sampling {num_samples} structures ({mode}, sampler {sampler}, {num_steps} steps, "
      f"batch size {size}, seed {seed})...")
start = time.time()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
while True:
    out_of_memory = False
    try:
        samples = session.sample(num_samples, batch_size=size, sampler=configured_sampler, seed=seed,
                                 progress=True, partial_dir=partial_dir)
        break
    except Exception as error:
        if not is_out_of_memory(error):
            raise
        out_of_memory = True  # Leaving the except block releases the failed batch's tensors.
    free_gpu_memory()
    if size == 1:
        raise RuntimeError(out_of_memory_advice("sampling with batch size 1"))
    if partial_dir is not None and partial_dir.exists():
        shutil.rmtree(partial_dir)  # Saved batches are tied to the old batch size.
    size = max(1, size // 2)
    pending["batch_size"] = size
    (run_dir / "pending.json").write_text(json.dumps(pending))
    print(f"Out of GPU memory; retrying with batch size {size}.")
elapsed = time.time() - start

report = check_physicality(samples, mode=physicality_check, system=SYSTEM)
save_frames(samples, run_dir, physicality=report)
write_system_yaml(run_dir / "system.yaml", SYSTEM, CONTACTS)
passed = int(report.passed.sum())
if report.checked and 0 < passed < samples.n_frames:
    save_frames(samples[np.flatnonzero(report.passed)], run_dir / "passed")
settings = {"task": task, "mode": mode, "num_samples": num_samples, "batch_size": size,
            "sampler": sampler, "step_scale": getattr(configured_sampler, "step_scale", None),
            "num_steps": num_steps, "seed": seed, "potentials_yaml": potentials_yaml.strip() or None,
            "potentials_sha256": potentials_sha, "custom_sampler_source": custom_source,
            "physicality_check": physicality_check,
            "contacts": [c.model_dump(exclude_none=True) for c in CONTACTS or []],
            **MODEL_INFO, "bioemu2": _version("bioemu2"), "elapsed_s": round(elapsed, 1)}
(run_dir / "run_info.json").write_text(json.dumps(settings, indent=2))
if partial_dir is not None and partial_dir.exists():
    shutil.rmtree(partial_dir)
(run_dir / "pending.json").unlink(missing_ok=True)
record = record_run(run_dir.name, kind="samples", mode=mode, frames=samples, directory=run_dir,
                    report=report, settings=settings)
drop_sessions()  # The embeddings stay cached for the next run; the session is not needed.

summary = [["run", record["name"]], ["structures", samples.n_frames],
           ["time", f"{elapsed:.0f} s ({elapsed / samples.n_frames:.1f} s per structure)"],
           ["GPU memory", gpu_memory_report()],
           ["physicality", physicality_summary(report)]]
rmsd = rmsd_to_reference(samples)
if rmsd is not None:
    summary.append(["CA RMSD to input", f"median {np.median(rmsd):.2f} Å (min {rmsd.min():.2f}, max {rmsd.max():.2f})"])
summary.append(["folder", str(run_dir)])
show_table(summary, ["", ""])

# 6. View the structures

Frames are superposed on the first frame. Drag to rotate, scroll to zoom. **Animation** cycles through frames; **overlay** shows them all at once, which highlights flexible regions; **grid** shows frames side by side, labelled with their run frame numbers; **single** shows one frame. Colour by `flexibility (RMSF)` to see which residues move most across the ensemble (blue rigid, red flexible).

Frame numbers are the run's own numbering everywhere: grid labels, `frame` below, `analysis.csv` (section 7) and the starting frames in 8.1 and 8.3.

In [ ]:
#@title 6. View structures
#@markdown `latest` (the newest run of the current system) or a run name from the list printed below.
run = "latest"  #@param {type:"string"}
layout = "animation"  #@param ["animation", "overlay", "grid", "single"]
color = "spectrum"  #@param ["spectrum", "chain", "secondary structure", "flexibility (RMSF)"]
frames_to_show = "physically passing frames"  #@param ["physically passing frames", "all frames"]
max_frames = 16  #@param {type:"slider", min:2, max:100, step:1}
#@markdown For `single`: the run frame number to show.
frame = 0  #@param {type:"integer"}

record = resolve_run(run)
list_runs()
if record["system_key"] != system_key(SYSTEM):
    print(f"Note: run '{record['name']}' belongs to an earlier system definition.")
if record["kind"] == "trajectory":
    print("This run holds several trajectories joined together; watch them one at a time with cell 8.2.")
trajectory, indices = select_frames(record, frames_to_show)
print(f"Showing {min(trajectory.n_frames, max_frames) if layout != 'single' else 1} of "
      f"{trajectory.n_frames} frames from run '{record['name']}'.")
show_ensemble(trajectory, frame_ids=indices, layout=layout, color=color, max_frames=max_frames,
              frame=frame).show()

# 7. Analyse the ensemble

Computes per-frame and per-residue metrics on the CA atoms (C4′ for nucleic acids):

* **RMSD to the reference**: the run's input structure when there is one (PDB ID or structure file), otherwise the medoid frame.
* **Radius of gyration** (all heavy atoms): compactness.
* **RMSF**: per-residue flexibility across the ensemble.
* **Pairwise RMSD**: structural diversity; blocks indicate distinct conformational states.
* **PCA**: the two main directions of conformational variation.
* **Secondary structure** (DSSP): fraction of frames in which each residue is helical or strand.

When there is an input structure, frames are superposed, and pairwise RMSD and PCA are computed, on the residues resolved in it, so disordered tags and termini do not dominate; RMSF is reported for all residues. Residues are numbered from 1 along each chain's full sequence, as in 3.1 and the output files; `rmsf.csv` also gives the author (PDB) numbering when the input file has it.

Figures and per-frame (`analysis.csv`) and per-residue (`rmsf.csv`) tables are saved in the run folder. With several runs of the same system, `compare_runs` overlays their RMSD and radius-of-gyration distributions against a shared reference.

In [ ]:
#@title 7. Analyse the ensemble
#@markdown `latest` (the newest run of the current system) or a run name.
run = "latest"  #@param {type:"string"}
frames_to_use = "all frames"  #@param ["all frames", "physically passing frames"]
compare_runs = True  #@param {type:"boolean"}

record = resolve_run(run)
trajectory, frame_ids = select_frames(record, frames_to_use)
indices = representative_indices(trajectory)
if len(indices) < 3:
    raise RuntimeError("This system has fewer than three polymer residues; nothing to analyse.")
reference, author = record.get("reference"), record.get("reference_author")
matched = matched_reference(trajectory, reference)
fit = matched[0] if matched is not None else indices
if matched is not None and len(fit) < len(indices):
    print(f"Superposition, pairwise RMSD and PCA use the {len(fit)} residues resolved in "
          f"{record['reference_file'].name}; {len(indices) - len(fit)} unresolved residues are excluded.")
aligned = trajectory[:]
aligned.superpose(aligned, frame=0, atom_indices=fit)

sub = aligned.atom_slice(fit)
medoid = int(np.argmin([mdtraj.rmsd(sub, sub, frame=i).sum() for i in range(sub.n_frames)])) \
    if sub.n_frames > 1 else 0
if matched is not None:
    rmsd_label = f"CA RMSD to {record['reference_file'].name} (Å)"
    rmsd_values = rmsd_to_reference(aligned, reference)
else:
    rmsd_label = f"CA RMSD to medoid frame {frame_ids[medoid]} (Å)"
    rmsd_values = 10 * mdtraj.rmsd(sub, sub, frame=medoid)
rg = 10 * mdtraj.compute_rg(aligned)
dssp = mdtraj.compute_dssp(aligned, simplified=True)
labels = residue_labels(aligned, indices, author)
residue_of = [aligned.topology.atom(i).residue.index for i in indices]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
ax = axes[0, 0]
ax.hist(rmsd_values, bins=min(30, max(5, len(rmsd_values) // 2)), color="#0078d4", alpha=0.85)
ax.set_xlabel(rmsd_label)
ax.set_ylabel("frames")
ax.set_title("RMSD distribution")

ax = axes[0, 1]
ax.hist(rg, bins=min(30, max(5, len(rg) // 2)), color="#2f9e44", alpha=0.85)
ax.set_xlabel("radius of gyration (Å)")
ax.set_title("Compactness")

ax = axes[0, 2]
if aligned.n_frames > 1:
    rmsf = rmsf_per_residue(aligned, indices, fit)
    ax.plot(rmsf, color="#e8590c", lw=1.2)
    for boundary in chain_boundaries(aligned, indices):
        ax.axvline(boundary, color="grey", ls="--", lw=0.8)
    ticks = np.linspace(0, len(labels) - 1, min(12, len(labels)), dtype=int)
    ax.set_xticks(ticks, [labels[t] for t in ticks], rotation=45, fontsize=8)
else:
    rmsf = np.zeros(len(indices))
    ax.text(0.5, 0.5, "needs at least 2 frames", ha="center", transform=ax.transAxes)
ax.set_ylabel("RMSF (Å)")
ax.set_title("Per-residue flexibility")

ax = axes[1, 0]
if aligned.n_frames > 1:
    shown, matrix = pairwise_rmsd(aligned, fit)
    image = ax.imshow(matrix, cmap="viridis", origin="lower")
    fig.colorbar(image, ax=ax, label="RMSD (Å)")
    ticks = np.linspace(0, len(shown) - 1, min(8, len(shown)), dtype=int)
    ax.set_xticks(ticks, frame_ids[shown[ticks]])
    ax.set_yticks(ticks, frame_ids[shown[ticks]])
    ax.set_xlabel("run frame")
    ax.set_ylabel("run frame")
else:
    ax.text(0.5, 0.5, "needs at least 2 frames", ha="center", transform=ax.transAxes)
ax.set_title("Pairwise CA RMSD")

ax = axes[1, 1]
if aligned.n_frames > 2:
    projection, variance = pca_projection(aligned, fit)
    points = ax.scatter(projection[:, 0], projection[:, 1], c=rmsd_values, cmap="plasma", s=25)
    fig.colorbar(points, ax=ax, label=rmsd_label.split(" (")[0])
    ax.set_xlabel(f"PC1 ({100 * variance[0]:.0f}% of variance)")
    ax.set_ylabel(f"PC2 ({100 * variance[1]:.0f}%)")
else:
    projection = np.zeros((aligned.n_frames, 2))
    ax.text(0.5, 0.5, "needs at least 3 frames", ha="center", transform=ax.transAxes)
ax.set_title("Principal components")

ax = axes[1, 2]
helix = (dssp[:, residue_of] == "H").mean(axis=0)
strand = (dssp[:, residue_of] == "E").mean(axis=0)
ax.fill_between(range(len(residue_of)), helix, color="#ae3ec9", alpha=0.7, label="helix", step="mid")
ax.fill_between(range(len(residue_of)), strand, color="#f59f00", alpha=0.7, label="strand", step="mid")
ax.set_ylim(0, 1)
ax.set_ylabel("fraction of frames")
ax.set_xlabel("residue (in order)")
ax.legend(loc="upper right", fontsize=8)
ax.set_title("Secondary structure (DSSP)")
fig.suptitle(f"Run '{record['name']}': {aligned.n_frames} frames", fontsize=13)
fig.tight_layout()
figure_path = record["dir"] / "analysis.png"
fig.savefig(figure_path, dpi=150)
plt.show()

report = record.get("report")
with open(record["dir"] / "analysis.csv", "w") as handle:
    handle.write("frame,rmsd_A,radius_of_gyration_A,pc1,pc2,physicality\n")
    for k, frame_id in enumerate(frame_ids):
        status = "not_checked"
        if report is not None and report.checked:
            status = "passed" if report.passed[frame_id] else (report.reasons[frame_id] or "rejected")
        handle.write(f"{frame_id},{rmsd_values[k]:.3f},{rg[k]:.3f},{projection[k, 0]:.3f},"
                     f"{projection[k, 1]:.3f},{status}\n")
with open(record["dir"] / "rmsf.csv", "w") as handle:
    handle.write("chain,sequence_position,residue_name,author_residue,rmsf_A,helix_fraction,strand_fraction\n")
    for k, i in enumerate(indices):
        residue = aligned.topology.atom(i).residue
        key = (residue.chain.chain_id, residue.resSeq)
        handle.write(f"{key[0]},{key[1]},{residue.name},{(author or {}).get(key, '')},"
                     f"{rmsf[k]:.3f},{helix[k]:.3f},{strand[k]:.3f}\n")

if report is not None:
    subset = type(report)(passed=report.passed[frame_ids], reasons=[report.reasons[f] for f in frame_ids],
                          mode=report.mode, checked=report.checked)
    print(f"Physicality: {physicality_summary(subset)}")
print(f"Saved {figure_path.name}, analysis.csv and rmsf.csv in {record['dir']}")

same_system = {name: r for name, r in RUNS.items()
               if r["kind"] == "samples" and r["system_key"] == record["system_key"]}
if compare_runs and len(same_system) > 1:
    reference_frame = aligned[medoid].atom_slice(fit)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for name, other in same_system.items():
        traj = other["frames"]
        values = (rmsd_to_reference(traj, reference) if matched is not None
                  else 10 * mdtraj.rmsd(traj.atom_slice(fit), reference_frame))
        axes[0].hist(values, bins=20, alpha=0.5, label=name, density=True)
        axes[1].hist(10 * mdtraj.compute_rg(traj), bins=20, alpha=0.5, label=name, density=True)
    axes[0].set_xlabel(rmsd_label if matched is not None
                       else f"CA RMSD to medoid frame {frame_ids[medoid]} of '{record['name']}' (Å)")
    axes[1].set_xlabel("radius of gyration (Å)")
    for ax in axes:
        ax.set_ylabel("density")
        ax.legend(fontsize=8)
    fig.suptitle("Comparison of runs of this system")
    fig.tight_layout()
    plt.show()

# 8. Simulate dynamics

BioEmu2 can **propagate** a structure forward in time: given a starting structure, it samples where the system will be after a time lag (`lag_time_ns`). Repeating this builds a **time-ordered trajectory**, frame *k* being at time *k* × lag. This is a learned emulation of molecular dynamics, not a physics-based simulation.

* Use lags of **1 to 1000 ns**, the range the model mostly saw in training. At much longer lags, endpoints approach the equilibrium ensemble.
* Starting structures come from a run of the current system in section 5 (equilibrium runs are the natural choice) or from the input structure. The input structure works only if every atom of the system's chains is resolved in it (its first model is used).
* Starting frame numbers are run frame numbers, as in the grid labels of section 6 and the `frame` column of `analysis.csv`.
* Only `edm` and `edm_gd` can build multi-step trajectories; resampling samplers mix up which trajectory a particle belongs to.
* Each trajectory step is a full denoising run, so a trajectory of 10 steps costs about as much as sampling a batch 10 times.
* Cell 8.3 instead draws independent endpoints at several lags from one starting frame, to show how far the system moves over time.

In [ ]:
#@title 8.1 Simulate trajectories
start_from = "a run from section 5"  #@param ["a run from section 5", "the input structure"]
#@markdown Run to take starting frames from (`latest` or a run name).
run = "latest"  #@param {type:"string"}
#@markdown Starting run frame numbers, e.g. `0` or `0, 3, 5`; `all` uses every frame (or the passing ones, below).
start_frames = "0"  #@param {type:"string"}
only_physically_passing = True  #@param {type:"boolean"}
#@markdown Independent trajectories per starting frame.
trajectories_per_start = 2  #@param {type:"integer"}
lag_time_ns = 1.0  #@param {type:"number"}
num_trajectory_steps = 10  #@param {type:"integer"}
sampler = "edm"  #@param ["edm", "edm_gd"]
num_steps = 200  #@param {type:"integer"}
batch_size = "auto"  #@param ["auto", "1", "2", "4", "8", "16", "32"] {allow-input: true}
seed = 0  #@param {type:"integer"}

import random

import biotite.structure.io.pdbx as pdbx

require("MODEL", "SYSTEM", "MSA_OBJ")
if not MODEL.supports_propagation:
    raise RuntimeError("The loaded checkpoint does not support propagation.")
if not 0 < lag_time_ns:
    raise ValueError("lag_time_ns must be positive.")
if not 1 <= lag_time_ns <= 1000:
    print(f"Note: lag_time_ns={lag_time_ns:g} is outside the 1–1000 ns range the model mostly saw in training.")
if num_trajectory_steps < 1 or trajectories_per_start < 1:
    raise ValueError("num_trajectory_steps and trajectories_per_start must be at least 1.")
if seed < 0:
    seed = random.randrange(2**31 - 1)

if start_from.startswith("a run"):
    source = resolve_run(run, kinds=("samples", "endpoints"))
    if source["system_key"] != system_key(SYSTEM):
        raise RuntimeError("That run belongs to an earlier system definition. Generate a run for the "
                           "current system first.")
    pool = source["frames"]
    report = source.get("report")
    if start_frames.strip().lower() == "all":
        chosen = np.arange(pool.n_frames)
        if only_physically_passing and report is not None and report.checked and report.passed.any():
            chosen = np.flatnonzero(report.passed)
    else:
        chosen = np.array([int(i) for i in split_list(start_frames)])
        if (chosen < 0).any() or (chosen >= pool.n_frames).any():
            raise IndexError(f"Starting frames must be between 0 and {pool.n_frames - 1}.")
        if only_physically_passing and report is not None and report.checked:
            failing = [int(i) for i in chosen if not report.passed[i]]
            if failing:
                print(f"Note: starting frame(s) {failing} failed the physicality check.")
    x0 = pool[np.repeat(chosen, trajectories_per_start)]
    origin = f"run '{source['name']}', frames {', '.join(map(str, chosen))}"
else:
    if REFERENCE_FILE is None or REFERENCE_FILE.suffix.lower() not in (".cif", ".mmcif"):
        raise RuntimeError("Starting from the input structure needs an mmCIF input (PDB ID or .cif file).")
    # Use the first model and only the system's chains (ligands or chains may have been left out).
    atoms = pdbx.get_structure(pdbx.CIFFile.read(str(REFERENCE_FILE)), model=1, use_author_fields=False)
    atoms = atoms[np.isin(atoms.chain_id, SYSTEM.chain_ids)]
    try:
        start_structure = map_cif_atom_order(SYSTEM, atoms)
    except Exception as error:
        raise RuntimeError(
            f"The input structure cannot be mapped onto the system: {error}\nStarting from it needs "
            "every atom of the system's chains; crystal structures often miss some. Start from a "
            "sampled run instead."
        ) from error
    x0 = start_structure[[0] * trajectories_per_start]
    chosen = np.array([0])
    origin = f"input structure {REFERENCE_FILE.name}"

configured_sampler = sampler_from_preset(sampler, mode="equilibrium", num_steps=num_steps)
size = (min(x0.n_frames, auto_batch_size(x0.n_frames, sampler)) if batch_size.strip().lower() == "auto"
        else int(batch_size))
embeddings = embeddings_for("equilibrium")
print(f"Simulating {x0.n_frames} trajectories of {num_trajectory_steps} steps × {lag_time_ns:g} ns "
      f"from {origin} (sampler {sampler}, batch size {size}, seed {seed})...")
start = time.time()
while True:
    out_of_memory = False
    try:
        trajectories = simulate_trajectory_with_embeddings(
            SYSTEM, embeddings, x0=x0, lag_time_ns=lag_time_ns, num_trajectory_steps=num_trajectory_steps,
            model=MODEL, batch_size=size, sampler=configured_sampler, seed=seed)
        break
    except Exception as error:
        if not is_out_of_memory(error):
            raise
        out_of_memory = True
    free_gpu_memory()
    if size == 1:
        raise RuntimeError(out_of_memory_advice("propagating with batch size 1"))
    size = max(1, size // 2)
    print(f"Out of GPU memory; retrying with batch size {size}.")
elapsed = time.time() - start
drop_sessions()

run_dir = unique_dir(JOB_DIR / f"dynamics_{lag_time_ns:g}ns_x{num_trajectory_steps}")
starts = np.repeat(chosen, trajectories_per_start)
reports = []
for index, trajectory in enumerate(trajectories):
    trajectory_report = check_physicality(trajectory, mode="high_quality", system=SYSTEM)
    reports.append(trajectory_report)
    save_frames(trajectory, run_dir / f"trajectory_{index:03d}", physicality=trajectory_report)
write_system_yaml(run_dir / "system.yaml", SYSTEM, CONTACTS)
settings = {"start_from": origin, "start_frames": starts.tolist(), "lag_time_ns": lag_time_ns,
            "num_trajectory_steps": num_trajectory_steps, "sampler": sampler, "num_steps": num_steps,
            "batch_size": size, "seed": seed, "elapsed_s": round(elapsed, 1)}
(run_dir / "run_info.json").write_text(json.dumps(settings, indent=2))
joined = mdtraj.join(trajectories)
combined_report = type(reports[0])(passed=np.concatenate([r.passed for r in reports]),
                                   reasons=[x for r in reports for x in r.reasons],
                                   mode=reports[0].mode, checked=reports[0].checked)
record = record_run(run_dir.name, kind="trajectory", mode="equilibrium", frames=joined, directory=run_dir,
                    report=combined_report, settings=settings, trajectories=trajectories)

time_ns = trajectories[0].time / 1000
representative = representative_indices(trajectories[0])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for index, trajectory in enumerate(trajectories):
    label = f"traj {index} (start {starts[index]})"
    values = rmsd_to_reference(trajectory)
    if values is None and len(representative) >= 3:
        sub = trajectory.atom_slice(representative)
        values = 10 * mdtraj.rmsd(sub, sub, frame=0)
    if values is not None:
        axes[0].plot(time_ns, values, marker=".", lw=1, label=label)
    axes[1].plot(time_ns, 10 * mdtraj.compute_rg(trajectory), marker=".", lw=1, label=label)
if len(representative) < 3:
    axes[0].text(0.5, 0.5, "no polymer residues", ha="center", transform=axes[0].transAxes)
axes[0].set_ylabel("CA RMSD to input structure (Å)" if REFERENCE else "CA RMSD to starting frame (Å)")
axes[1].set_ylabel("radius of gyration (Å)")
for ax in axes:
    ax.set_xlabel("time (ns)")
if len(trajectories) <= 10:
    axes[1].legend(fontsize=8)
fig.suptitle(f"{len(trajectories)} trajectories, lag {lag_time_ns:g} ns")
fig.tight_layout()
fig.savefig(run_dir / "timeseries.png", dpi=150)
plt.show()
print(f"Done in {elapsed:.0f} s. Physicality ({joined.n_frames} frames): {physicality_summary(combined_report)}. "
      f"Saved to {run_dir} as run '{record['name']}'.")

In [ ]:
#@title 8.2 Watch a trajectory
#@markdown `latest` or the name of a dynamics run.
run = "latest"  #@param {type:"string"}
trajectory_index = 0  #@param {type:"integer"}
color = "spectrum"  #@param ["spectrum", "chain", "secondary structure", "flexibility (RMSF)"]
max_frames = 50  #@param {type:"slider", min:2, max:200, step:1}
#@markdown Milliseconds per frame.
interval = 300  #@param {type:"slider", min:50, max:2000, step:50}

record = resolve_run(run, kinds=("trajectory",))
trajectories = record["trajectories"]
if not 0 <= trajectory_index < len(trajectories):
    raise IndexError(f"trajectory_index must be between 0 and {len(trajectories) - 1}.")
trajectory = trajectories[trajectory_index]
print(f"Run '{record['name']}', trajectory {trajectory_index}: {trajectory.n_frames} frames, "
      f"{trajectory.time[-1] / 1000:g} ns (showing at most {max_frames} evenly spaced frames).")
show_ensemble(trajectory, layout="animation", color=color, max_frames=max_frames, interval=interval).show()

In [ ]:
#@title 8.3 Endpoints at several lag times
#@markdown From one starting frame, draw independent endpoints at each lag time to see how far the system moves over time. Unlike 8.1, these are not consecutive frames of one trajectory. All four samplers work here, because every endpoint starts from the same frame.
#@markdown `latest` or the name of a run from section 5.
run = "latest"  #@param {type:"string"}
#@markdown Run frame number to start from.
start_frame = 0  #@param {type:"integer"}
lag_times_ns = "1, 10, 100, 1000"  #@param {type:"string"}
endpoints_per_lag = 8  #@param {type:"integer"}
sampler = "edm"  #@param ["edm", "edm_gd", "edm_fk", "edm_fkc"]
num_steps = 200  #@param {type:"integer"}
batch_size = "auto"  #@param ["auto", "1", "2", "4", "8", "16", "32"] {allow-input: true}
seed = 0  #@param {type:"integer"}

import random

require("MODEL", "SYSTEM", "MSA_OBJ")
if not MODEL.supports_propagation:
    raise RuntimeError("The loaded checkpoint does not support propagation.")
lags = [float(value) for value in split_list(lag_times_ns)]
if not lags or min(lags) <= 0:
    raise ValueError("Enter one or more positive lag times, e.g. '1, 10, 100'.")
if endpoints_per_lag < 1:
    raise ValueError("endpoints_per_lag must be at least 1.")
if seed < 0:
    seed = random.randrange(2**31 - 1)
source = resolve_run(run, kinds=("samples",))
if source["system_key"] != system_key(SYSTEM):
    raise RuntimeError("That run belongs to an earlier system definition. Generate a run for the current system first.")
pool = source["frames"]
if not 0 <= start_frame < pool.n_frames:
    raise IndexError(f"start_frame must be between 0 and {pool.n_frames - 1}.")
x0 = pool[[start_frame] * endpoints_per_lag]
if batch_size.strip().lower() == "auto":
    # Resampling samplers treat the batch as one population, so keep all endpoints of a lag together.
    size = endpoints_per_lag if sampler in ("edm_fk", "edm_fkc") else min(endpoints_per_lag, auto_batch_size(endpoints_per_lag, sampler))
else:
    size = int(batch_size)
session = EmuSession(SYSTEM, model=MODEL, mode="equilibrium", msa=MSA_OBJ, contacts=CONTACTS,
                     embeddings=embeddings_for("equilibrium"))
run_dir = unique_dir(JOB_DIR / f"endpoints_from_{source['dir'].name}_frame{start_frame}")
indices = representative_indices(pool)
has_rmsd = len(indices) >= 3
start_structure = pool[start_frame].atom_slice(indices) if has_rmsd else None
all_endpoints, rows, distances = [], [], {}
start = time.time()
for k, lag in enumerate(lags):
    sampler_object = sampler_from_preset(sampler, mode="equilibrium", num_steps=num_steps)
    while True:
        out_of_memory = False
        try:
            endpoints = session.propagate(x0, lag_time_ns=lag, batch_size=size, sampler=sampler_object,
                                          seed=seed + 1_000_000 * k, progress=False)
            break
        except Exception as error:
            if not is_out_of_memory(error):
                raise
            out_of_memory = True
        free_gpu_memory()
        if size == 1:
            raise RuntimeError(out_of_memory_advice("propagating with batch size 1"))
        size = max(1, size // 2)
        print(f"Out of GPU memory; retrying with batch size {size}.")
    lag_report = check_physicality(endpoints, mode="high_quality", system=SYSTEM)
    save_frames(endpoints, run_dir / f"lag_{lag:g}ns", physicality=lag_report)
    reference_rmsd = rmsd_to_reference(endpoints)
    if has_rmsd:
        distances[lag] = 10 * mdtraj.rmsd(endpoints.atom_slice(indices), start_structure)
    rows.append([f"{lag:g} ns", endpoints.n_frames,
                 f"{np.median(distances[lag]):.2f} Å" if has_rmsd else "–",
                 f"{np.median(reference_rmsd):.2f} Å" if reference_rmsd is not None else "–",
                 f"{int(lag_report.passed.sum())}/{endpoints.n_frames}" if lag_report.checked else "not checked"])
    all_endpoints.append((endpoints, lag_report))
    print(f"lag {lag:g} ns done ({time.time() - start:.0f} s)")
drop_sessions()

frames = mdtraj.join([e for e, _ in all_endpoints])
report = type(all_endpoints[0][1])(passed=np.concatenate([r.passed for _, r in all_endpoints]),
                                   reasons=[x for _, r in all_endpoints for x in r.reasons],
                                   mode=all_endpoints[0][1].mode, checked=all_endpoints[0][1].checked)
settings = {"source_run": source["name"], "start_frame": start_frame, "lag_times_ns": lags,
            "endpoints_per_lag": endpoints_per_lag, "sampler": sampler, "num_steps": num_steps,
            "batch_size": size, "seed": seed, "frame_lags_ns": [lag for lag in lags for _ in range(endpoints_per_lag)]}
(run_dir / "run_info.json").write_text(json.dumps(settings, indent=2))
write_system_yaml(run_dir / "system.yaml", SYSTEM, CONTACTS)
record = record_run(run_dir.name, kind="endpoints", mode="equilibrium", frames=frames, directory=run_dir,
                    report=report, settings=settings)

show_table(rows, ["lag", "endpoints", "median CA RMSD to start", "median CA RMSD to input", "physicality"])
if has_rmsd:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.boxplot([distances[lag] for lag in lags], positions=range(len(lags)), widths=0.5)
    ax.set_xticks(range(len(lags)), [f"{lag:g}" for lag in lags])
    ax.set_xlabel("lag time (ns)")
    ax.set_ylabel("CA RMSD to starting frame (Å)")
    ax.set_title(f"Endpoints from frame {start_frame} of '{source['name']}'")
    fig.tight_layout()
    fig.savefig(run_dir / "rmsd_vs_lag.png", dpi=150)
    plt.show()
print(f"Saved to {run_dir}. View the endpoints with section 6 (run '{record['name']}').")

# 9. Save and download

Every run is already saved as `samples.xtc` + `topology.cif` (open them in PyMOL, ChimeraX or VMD, or load them with MDTraj). This cell also writes multi-model **PDB** files, which many tools open directly, and packs the job folder into a zip file to download.

PDB chain IDs are the chain IDs shown in 3.1. If any chain ID is longer than one character (for example `A-2` in a biological assembly), chains are relabelled A, B, C… in order, and `chain_map.tsv` lists the mapping. If Google Drive was enabled in 1.3 before the job's runs were made, the files are already on your Drive.

In [ ]:
#@title 9. Save and download results
what = "the whole job folder"  #@param ["the whole job folder", "the latest run only"]
write_pdb_files = True  #@param {type:"boolean"}
download_zip = True  #@param {type:"boolean"}

require("SYSTEM")
if not RUNS:
    raise RuntimeError("No results yet. Run section 5 first.")
if what.startswith("the whole"):
    selected = {name: r for name, r in RUNS.items() if JOB_DIR in r["dir"].parents}
    folders = [JOB_DIR]
    archive_name = JOB_NAME
else:
    latest = RUNS[LAST_RUN]
    selected = {LAST_RUN: latest}
    folders = [latest["dir"]]
    archive_name = f"{latest['dir'].parent.name}_{latest['dir'].name}"
if not selected:
    raise RuntimeError(f"No runs of job '{JOB_NAME}' yet.")

if write_pdb_files:
    for name, record in selected.items():
        try:
            groups = record["trajectories"] if record["kind"] == "trajectory" else [record["frames"]]
            for index, trajectory in enumerate(groups):
                folder = record["dir"] / (f"trajectory_{index:03d}" if record["kind"] == "trajectory" else "")
                aligned = trajectory[:]
                fit = representative_indices(aligned)
                aligned.superpose(aligned, frame=0, atom_indices=fit if len(fit) >= 3 else None)
                (folder / "models.pdb").write_text(pdb_text(aligned))
                report = record.get("report")
                if record["kind"] != "trajectory" and report is not None and report.checked and report.passed.any():
                    (folder / "models_passed.pdb").write_text(pdb_text(aligned[np.flatnonzero(report.passed)]))
                letters = pdb_chain_ids(aligned.topology)
                canonical = [chain.chain_id for chain in aligned.topology.chains]
                if letters != canonical:
                    author = SYSTEM.asym_id_to_author_id or {}
                    (folder / "chain_map.tsv").write_text("pdb_chain\tchain\tauthor_chain\n" + "".join(
                        f"{l}\t{c}\t{author.get(c, '')}\n" for l, c in zip(letters, canonical)))
            print(f"Wrote PDB files for run '{name}'.")
        except Exception as error:  # One unusual run should not stop the archive.
            print(f"Could not write PDB files for run '{name}': {error}")

archive_dir = Path("/content" if Path("/content").is_dir() else Path.home())
archive = archive_dir / f"{archive_name}_{datetime.now():%Y%m%d_%H%M}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
    for folder in folders:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and "partial" not in path.parts:
                handle.write(path, path.relative_to(folder.parent))
size_mb = archive.stat().st_size / 1024**2
print(f"Created {archive} ({size_mb:.1f} MB).")

rows = []
for name, record in selected.items():
    for path in sorted(record["dir"].rglob("*")):
        if path.is_file():
            rows.append([name, str(path.relative_to(record["dir"])), f"{path.stat().st_size / 1024:.0f} KB"])
show_table(rows[:60], ["run", "file", "size"])
if len(rows) > 60:
    print(f"... and {len(rows) - 60} more files.")

if download_zip:
    try:
        from google.colab import files

        files.download(str(archive))
    except ImportError:
        print("Automatic download needs Google Colab; the zip file is at the path above.")

# 10. Command-line interface (optional)

Everything above uses the Python API. BioEmu2 also has a command-line interface with the same capabilities, convenient for scripted or long runs. The cell below runs a `bioemu2` command inside the outputs folder shown in 1.3, so relative paths start there (and with Google Drive enabled, results land on Drive). It also prints ready-to-run commands for the latest run. Some examples:

```bash
bioemu2 --help
bioemu2 sample-equilibrium --pdb-id 1AKE --num-samples 32 --batch-size 16 --out cli_adk
bioemu2 predict-structure --pdb-id 1UBQ --num-samples 4 --out cli_ubq
bioemu2 sample-equilibrium --system MYJOB/input/system.yaml --sampler edm_gd --out cli_steered
bioemu2 propagate --system MYJOB/MYRUN/system.yaml --x0-cif MYJOB/MYRUN/topology.cif --x0-xtc MYJOB/MYRUN/samples.xtc --lag-time-ns 10 --out cli_prop
```

Replace `MYJOB` and `MYRUN` with a job and run folder. Topology files written by this notebook do not contain the entity records the CLI needs to infer a system, so always pass the run's `system.yaml` with `--system` when using notebook outputs. `--pdb-id 1AKE` keeps the Ap5A inhibitor, as in 3.1. Use a new `--out` folder for each command. The command runs in a separate process that loads its own copy of the model; on a small GPU, tick `release_notebook_model` first.

In [ ]:
#@title 10. Run a BioEmu2 command
command = "bioemu2 --help"  #@param {type:"string"}
#@markdown Free the notebook's model and embeddings first, so the command has the whole GPU (re-run section 2 afterwards).
release_notebook_model = False  #@param {type:"boolean"}

import shlex
import subprocess

if "OUTPUT_ROOT" not in globals():
    raise RuntimeError("Run cell 1.3 first, so the command uses the same caches as the notebook.")
if globals().get("RUNS") and globals().get("LAST_RUN") in RUNS:
    latest = RUNS[LAST_RUN]
    folder = latest["dir"].relative_to(OUTPUT_ROOT) if OUTPUT_ROOT in latest["dir"].parents else latest["dir"]
    print("Commands for the latest run:")
    if latest["kind"] != "trajectory":
        print(f"  bioemu2 propagate --system {folder}/system.yaml --x0-cif {folder}/topology.cif "
              f"--x0-xtc {folder}/samples.xtc --lag-time-ns 10 --out cli_prop_{latest['dir'].name}")
    print(f"  bioemu2 sample-equilibrium --system {folder}/system.yaml --num-samples 32 --batch-size 16 "
          f"--out cli_{latest['dir'].name}\n")
arguments = shlex.split(command)
if not arguments or arguments[0] != "bioemu2":
    raise ValueError("The command must start with 'bioemu2'.")
if "free_gpu_memory" in globals() and not ({"--help", "-h"} & set(arguments)):
    if release_notebook_model:
        drop_sessions()
        MODEL = None
        EMBEDDINGS.clear()
    free_gpu_memory()
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"{free / 1024**3:.1f} GB of {total / 1024**3:.0f} GB GPU memory free for the command.")
process = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                           cwd=str(OUTPUT_ROOT))
for line in process.stdout:
    print(line, end="")
if process.wait() != 0:
    raise RuntimeError(f"The command exited with status {process.returncode}.")

# 11. Troubleshooting

**No GPU / "No NVIDIA GPU was found".** Runtime → Change runtime type → choose a GPU, then run all cells again. Free-tier GPUs are not always available; try again later or use a paid runtime.

**Installation fails or BioEmu2 fails to import.** Follow the advice printed by cell 1.2; it depends on the GPU. In short: re-run 1.2 with `force_reinstall` ticked and `gpu_build = "CUDA 13"`, or on T4 and V100 GPUs `"CUDA 12.6"` (the configuration the BioEmu2 authors tested on T4). Blackwell GPUs (G4) need CUDA 13. Runtime → Disconnect and delete runtime restores Colab's own packages. If Colab's Python version is unsupported, change Runtime → Change runtime type → Runtime version.

**"The runtime restarted" / the session crashed after 1.2.** This is intentional when the installation upgraded a package that was already loaded, or reinstalled BioEmu2 after it was imported. Continue from cell 1.3, or use Runtime → Run all again (installation is skipped).

**`no kernel image is available` or "not supported by the installed PyTorch".** The installed PyTorch has no kernels for this GPU; section 2 prints which build to install in 1.2.

**Errors from cuEquivariance, Triton or `cueq_torch` during trunk embeddings** (for example `NameError: name 'cueq_torch' is not defined`, or a Triton compilation error). Re-run 1.2 with another `gpu_build` and `force_reinstall` (on a T4, `CUDA 12.6` is the tested configuration), then try `precision = "32-true"` in section 2. If the kernels still fail, choose `kernels = "pure PyTorch"` in section 2. Systems of 100 tokens or fewer never use these kernels, so a small test system can succeed while a larger one fails. If Triton reports that it cannot find `libcuda.so`, re-run cell 1.2, which registers Colab's driver directory.

**Out of GPU memory.** During sampling the notebook halves the batch size automatically. If it fails while *computing trunk embeddings*, the system is too large for this GPU as configured: keep `precision = "bf16-mixed"` and try `kernels = "memory-saving"` (section 2), set `max_msa_rows` to 1024 or 512 (section 4), drop chains or ligands you do not need (section 3), or switch to a runtime with more GPU memory (L4: 24 GB, A100: 40/80 GB, H100: 80 GB, G4: 96 GB). Steering samplers need more memory than `edm`, and `edm_fkc` the most. The notebook releases memory held by failed attempts; `free_gpu_memory()` in a cell does the same. A command from section 10 needs its own GPU memory.

**The ColabFold MSA search fails or hangs.** The public server is shared and rate-limited, and the client waits indefinitely for a queued job. Interrupt the cell (■) after about 15 minutes, wait a few minutes and retry (completed searches are cached), upload A3M files from another tool, or use single-sequence mode as a last resort.

**Checkpoint download fails (401/403).** Add a Hugging Face token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar), grant this notebook access, and re-run 1.3 and 2.

**"Run section X first."** Cells depend on earlier ones. After changing the system (3.1), also re-run 3.2 if you use contacts, and 4 (MSA), before generating structures. `latest` in sections 6–9 means the newest run of the current system.

**Uploaded PDB file: residues are missing or chains are merged.** PDB files do not record unresolved residues, so gaps are joined. Prefer mmCIF files with full sequence records (for example from RCSB or PDB-REDO), or enter the full sequences directly. The file's own chain IDs appear in the "author chain" column of 3.1.

**Ligand not recognised.** Ligands are defined by RCSB Chemical Component Dictionary codes (e.g. `ATP`, `HEM`, `MG`) or SMILES. Covalent attachments and glycans need a system YAML; see the [system guide](https://github.com/microsoft/bioemu2/blob/main/docs/system.md).

**Disconnected during a long run.** With `resumable` ticked in 5.2, finished batches are kept in the run folder. After reconnecting, run cells 1.1 to 4 again with the same inputs and job name, then 5.2 with the same settings: it finds the unfinished run and continues it with its original seed and batch size. With Google Drive enabled before the run started, this also works in a new session.

**Large systems that do not fit on one GPU.** BioEmu2 supports context-parallel sampling across four or more GPUs, which Colab does not offer; see the [context-parallel guide](https://github.com/microsoft/bioemu2/blob/main/docs/context-parallel-sampling.md).

---

**License and citation.** BioEmu2 is released under the [MIT License](https://github.com/microsoft/bioemu2/blob/main/LICENSE); see the repository for third-party notices and how to cite it. If you use MSAs from the ColabFold server, also cite: Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. *ColabFold: making protein folding accessible to all.* Nature Methods 19, 679–682 (2022).